In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os


# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install vegafusion vegafusion-python

In [ ]:
import altair as alt

In [ ]:
df=pd.read_csv("../data/rideshare_kaggle.csv")
df.head()

In [ ]:
df.info()

In [ ]:
# eda check for null values
for i in df.columns:
    if df[i].isnull().sum()>0:
        print(f"{i}: {df[i].isnull().sum()}: % missing : {(df[i].isnull().sum()/len(df[i]))*100}")

In [ ]:
# Sanity check for checking bias for price
print(f"{df[df['price'].isna()]['cab_type'].value_counts()}")
print("----------------------------------------------------")
print(f"{df[df['price'].isna()]['source'].value_counts()}")
print("----------------------------------------------------")
print(f"{df[df['price'].isna()]['surge_multiplier'].value_counts()}")
print("----------------------------------------------------")
print(f"{df[df['price'].isna()]['name'].value_counts()}")
print("----------------------------------------------------")

#sanity checks for the other major columns i can think of 
print(f"{df[df['price'] <= 0].shape}")
print(f"{df['price'].describe()}")
print("---------------------------------------------------")
print(f"{df['surge_multiplier'].value_counts()}")
print("---------------------------------------------------")
print(f"{df[df['distance'] <= 0].shape}")
print("---------------------------------------------------")
print(f"{df.groupby(['cab_type', 'name'])['price'].count()}")

In [ ]:
df.dropna(axis=0,inplace=True)

In [ ]:
alt.data_transformers.enable("vegafusion")  

In [ ]:
hist = (
    alt.Chart(df)
    .mark_bar(opacity=0.7, color="#4C78A8")
    .encode(
        alt.X(
            "price:Q",
            bin=alt.Bin(maxbins=60),
            title="Price ($)",
            axis=alt.Axis(labelFontSize=12, titleFontSize=13),
        ),
        alt.Y(
            "count():Q",
            title="Number of Rides",
            axis=alt.Axis(labelFontSize=12, titleFontSize=13),
        ),
        tooltip=[
            alt.Tooltip("price:Q", bin=alt.Bin(maxbins=60), title="Price Range"),
            alt.Tooltip("count():Q", title="Rides"),
        ],
    )
    .properties(
        width=650,
        height=380,
        title=alt.TitleParams(
            "Distribution of Ride Prices",
            fontSize=16,
            fontWeight="bold",
            anchor="start",
            subtitle="Boston Uber & Lyft | Nov–Dec 2018  •  n = 637,976",
            subtitleFontSize=12,
            subtitleColor="#666",
        ),
    )
)

# Mean & Median reference lines
mean_price = df["price"].mean()
median_price = df["price"].median()

mean_line = (
    alt.Chart(pd.DataFrame({"price": [mean_price]}))
    .mark_rule(color="#E45756", strokeDash=[6, 3], strokeWidth=2)
    .encode(x="price:Q", tooltip=[alt.Tooltip("price:Q", title="Mean", format=".2f")])
)

median_line = (
    alt.Chart(pd.DataFrame({"price": [median_price]}))
    .mark_rule(color="#F58518", strokeDash=[6, 3], strokeWidth=2)
    .encode(x="price:Q", tooltip=[alt.Tooltip("price:Q", title="Median", format=".2f")])
)

mean_label = (
    alt.Chart(pd.DataFrame({"price": [mean_price], "y": [18000], "label": [f"Mean ${mean_price:.2f}"]}))
    .mark_text(align="left", dx=6, dy=0, color="#E45756", fontSize=11, fontWeight="bold")
    .encode(x="price:Q", y=alt.value(30), text="label:N")
)

median_label = (
    alt.Chart(pd.DataFrame({"price": [median_price], "label": [f"Median ${median_price:.2f}"]}))
    .mark_text(align="left", dx=6, dy=0, color="#F58518", fontSize=11, fontWeight="bold")
    .encode(x="price:Q", y=alt.value(50), text="label:N")
)

chart1 = (hist + mean_line + median_line + mean_label + median_label).configure_view(
    strokeWidth=0
).configure_axis(
    grid=True, gridColor="#e8e8e8", gridOpacity=0.6
)

In [ ]:
hist

In [ ]:
color_scale = alt.Scale(
    domain=["Uber", "Lyft"],
    range=["#1a1a2e", "#E91E8C"]   # Uber black-navy vs Lyft magenta
)

boxplot = (
    alt.Chart(df)
    .mark_boxplot(size=60, outliers=alt.MarkConfig(size=8, opacity=0.15))
    .encode(
        alt.X(
            "cab_type:N",
            title="Cab Type",
            axis=alt.Axis(labelFontSize=13, titleFontSize=13, labelAngle=0),
        ),
        alt.Y(
            "price:Q",
            title="Price ($)",
            scale=alt.Scale(zero=False),
            axis=alt.Axis(labelFontSize=12, titleFontSize=13),
        ),
        alt.Color("cab_type:N", scale=color_scale, legend=None),
        tooltip=[
            alt.Tooltip("cab_type:N", title="Type"),
            alt.Tooltip("mean(price):Q", title="Avg Price", format="$.2f"),
        ],
    )
    .properties(
        width=400,
        height=380,
        title=alt.TitleParams(
            "Price Distribution: Uber vs Lyft",
            fontSize=16,
            fontWeight="bold",
            anchor="start",
            subtitle="Boxplot with outliers  •  IQR + whiskers",
            subtitleFontSize=12,
            subtitleColor="#666",
        ),
    )
)

# Overlay mean dots
mean_dots = (
    alt.Chart(df)
    .mark_point(shape="diamond", size=120, filled=True, color="white", strokeWidth=1.5)
    .encode(
        x=alt.X("cab_type:N"),
        y=alt.Y("mean(price):Q"),
        stroke=alt.Stroke("cab_type:N", scale=color_scale),
        tooltip=[
            alt.Tooltip("cab_type:N", title="Type"),
            alt.Tooltip("mean(price):Q", title="Mean Price", format="$.2f"),
        ],
    )
)

chart2 = (boxplot + mean_dots).configure_view(strokeWidth=0).configure_axis(
    grid=True, gridColor="#e8e8e8", gridOpacity=0.6
)


In [ ]:
tier_order = (
    df.groupby("name")["price"]
    .median()
    .sort_values()
    .index.tolist()
)

color_scale = alt.Scale(
    domain=["Lyft", "Uber"],
    range=["#E91E8C", "#1a1a2e"]
)

boxplot3 = (
    alt.Chart(df)
    .mark_boxplot(size=30, outliers=alt.MarkConfig(size=6, opacity=0.1))
    .encode(
        alt.X(
            "name:N",
            title="Ride Tier",
            sort=tier_order,
            axis=alt.Axis(labelFontSize=11, titleFontSize=13, labelAngle=-30),
        ),
        alt.Y(
            "price:Q",
            title="Price ($)",
            scale=alt.Scale(zero=False),
            axis=alt.Axis(labelFontSize=11, titleFontSize=13),
        ),
        alt.Color("cab_type:N", scale=color_scale, legend=alt.Legend(title="Cab Type")),
        tooltip=[
            alt.Tooltip("name:N", title="Tier"),
            alt.Tooltip("cab_type:N", title="Brand"),
            alt.Tooltip("mean(price):Q", title="Avg Price", format="$.2f"),
            alt.Tooltip("median(price):Q", title="Median Price", format="$.2f"),
        ],
    )
    .properties(
        width=680,
        height=400,
        title=alt.TitleParams(
            "Price Distribution by Ride Tier",
            fontSize=16,
            fontWeight="bold",
            anchor="start",
            subtitle="Tiers sorted by median price  •  Uber (dark) vs Lyft (pink)",
            subtitleFontSize=12,
            subtitleColor="#666",
        ),
    )
)

mean_dots3 = (
    alt.Chart(df)
    .mark_point(shape="diamond", size=80, filled=True, color="white", strokeWidth=1.2)
    .encode(
        x=alt.X("name:N", sort=tier_order),
        y=alt.Y("mean(price):Q"),
        stroke=alt.Stroke("cab_type:N", scale=color_scale),
        tooltip=[
            alt.Tooltip("name:N", title="Tier"),
            alt.Tooltip("mean(price):Q", title="Mean Price", format="$.2f"),
        ],
    )
)

chart3 = (
    (boxplot3 + mean_dots3)
    .configure_view(strokeWidth=0)
    .configure_axis(grid=True, gridColor="#e8e8e8", gridOpacity=0.6)
    .configure_legend(labelFontSize=11, titleFontSize=12)
)
chart3

In [ ]:
df_sample = df.sample(30000, random_state=42)

# Scatter
scatter4 = (
    alt.Chart(df_sample)
    .mark_point(opacity=0.15, size=18)
    .encode(
        alt.X(
            "distance:Q",
            title="Distance (miles)",
            scale=alt.Scale(domain=[0, df["distance"].quantile(0.99)]),
            axis=alt.Axis(labelFontSize=11, titleFontSize=13),
        ),
        alt.Y(
            "price:Q",
            title="Price ($)",
            scale=alt.Scale(domain=[0, df["price"].quantile(0.99)]),
            axis=alt.Axis(labelFontSize=11, titleFontSize=13),
        ),
        alt.Color(
            "cab_type:N",
            scale=alt.Scale(domain=["Uber", "Lyft"], range=["#1a1a2e", "#E91E8C"]),
            legend=alt.Legend(title="Cab Type"),
        ),
        tooltip=[
            alt.Tooltip("distance:Q", title="Distance (mi)", format=".2f"),
            alt.Tooltip("price:Q", title="Price ($)", format="$.2f"),
            alt.Tooltip("cab_type:N", title="Brand"),
            alt.Tooltip("name:N", title="Tier"),
        ],
    )
)

# Regression lines — one per cab type
regression_uber = (
    alt.Chart(df_sample[df_sample["cab_type"] == "Uber"])
    .transform_regression("distance", "price")
    .mark_line(strokeWidth=2.5, color="#1a1a2e", strokeDash=[1, 0])
    .encode(
        x="distance:Q",
        y="price:Q",
    )
)

regression_lyft = (
    alt.Chart(df_sample[df_sample["cab_type"] == "Lyft"])
    .transform_regression("distance", "price")
    .mark_line(strokeWidth=2.5, color="#E91E8C", strokeDash=[1, 0])
    .encode(
        x="distance:Q",
        y="price:Q",
    )
)

chart4 = (
    (scatter4 + regression_uber + regression_lyft)
    .properties(
        width=680,
        height=420,
        title=alt.TitleParams(
            "Price vs Distance with Regression Lines",
            fontSize=16,
            fontWeight="bold",
            anchor="start",
            subtitle="30K sampled rides  •  Regression fitted separately for Uber & Lyft  •  99th percentile axes",
            subtitleFontSize=12,
            subtitleColor="#666",
        ),
    )
    .configure_view(strokeWidth=0)
    .configure_axis(grid=True, gridColor="#e8e8e8", gridOpacity=0.6)
    .configure_legend(labelFontSize=11, titleFontSize=12)
)
chart4

In [ ]:
df["datetime"] = pd.to_datetime(df["datetime"])
df["hour"] = df["datetime"].dt.hour
df["day_of_week"] = df["datetime"].dt.day_name()

hourly = (
    df.groupby(["hour", "cab_type"])["price"]
    .mean()
    .reset_index()
    .rename(columns={"price": "avg_price"})
)

color_scale = alt.Scale(
    domain=["Uber", "Lyft"],
    range=["#1a1a2e", "#E91E8C"]
)

# Line + points for avg price per hour
line5 = (
    alt.Chart(hourly)
    .mark_line(strokeWidth=2.5, point=alt.OverlayMarkDef(filled=True, size=60))
    .encode(
        alt.X(
            "hour:Q",
            title="Hour of Day (0 = Midnight)",
            axis=alt.Axis(
                labelFontSize=11,
                titleFontSize=13,
                values=list(range(0, 24)),
                labelExpr="datum.value + ':00'",
            ),
        ),
        alt.Y(
            "avg_price:Q",
            title="Avg Price ($)",
            scale=alt.Scale(zero=False),
            axis=alt.Axis(labelFontSize=11, titleFontSize=13, format="$.2f"),
        ),
        alt.Color("cab_type:N", scale=color_scale, legend=alt.Legend(title="Cab Type")),
        tooltip=[
            alt.Tooltip("hour:Q", title="Hour"),
            alt.Tooltip("avg_price:Q", title="Avg Price", format="$.2f"),
            alt.Tooltip("cab_type:N", title="Brand"),
        ],
    )
)

# Regression line over hourly aggregated data — one per cab type
reg_uber5 = (
    alt.Chart(hourly[hourly["cab_type"] == "Uber"])
    .transform_regression("hour", "avg_price")
    .mark_line(strokeWidth=2, color="#1a1a2e", strokeDash=[6, 3], opacity=0.6)
    .encode(x="hour:Q", y="avg_price:Q")
)

reg_lyft5 = (
    alt.Chart(hourly[hourly["cab_type"] == "Lyft"])
    .transform_regression("hour", "avg_price")
    .mark_line(strokeWidth=2, color="#E91E8C", strokeDash=[6, 3], opacity=0.6)
    .encode(x="hour:Q", y="avg_price:Q")
)

# Shade peak hours (7-9am, 5-8pm)
peak_am = (
    alt.Chart(pd.DataFrame({"x1": [7], "x2": [9]}))
    .mark_rect(opacity=0.07, color="#F58518")
    .encode(x="x1:Q", x2="x2:Q")
)

peak_pm = (
    alt.Chart(pd.DataFrame({"x1": [17], "x2": [20]}))
    .mark_rect(opacity=0.07, color="#F58518")
    .encode(x="x1:Q", x2="x2:Q")
)

peak_label_am = (
    alt.Chart(pd.DataFrame({"x": [8], "y": [22], "label": ["AM Peak"]}))
    .mark_text(fontSize=10, color="#F58518", fontWeight="bold", angle=0)
    .encode(x="x:Q", y=alt.value(20), text="label:N")
)

peak_label_pm = (
    alt.Chart(pd.DataFrame({"x": [18.5], "label": ["PM Peak"]}))
    .mark_text(fontSize=10, color="#F58518", fontWeight="bold", angle=0)
    .encode(x="x:Q", y=alt.value(20), text="label:N")
)

chart5 = (
    (peak_am + peak_pm + line5 + reg_uber5 + reg_lyft5 + peak_label_am + peak_label_pm)
    .properties(
        width=700,
        height=400,
        title=alt.TitleParams(
            "Average Ride Price by Hour of Day",
            fontSize=16,
            fontWeight="bold",
            anchor="start",
            subtitle="Dashed lines = linear regression trend  •  Shaded = typical peak commute hours",
            subtitleFontSize=12,
            subtitleColor="#666",
        ),
    )
    .configure_view(strokeWidth=0)
    .configure_axis(grid=True, gridColor="#e8e8e8", gridOpacity=0.6)
    .configure_legend(labelFontSize=11, titleFontSize=12)
)
chart5

In [ ]:
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

dday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

daily = (
    df.groupby(["day_of_week", "cab_type"])["price"]
    .mean()
    .reset_index()
    .rename(columns={"price": "avg_price"})
)

print(daily.sort_values("day_of_week"))
print(f"\nMin: {daily['avg_price'].min():.2f}, Max: {daily['avg_price'].max():.2f}")

color_scale = alt.Scale(
    domain=["Uber", "Lyft"],
    range=["#1a1a2e", "#E91E8C"]
)

bars6 = (
    alt.Chart(daily)
    .mark_bar(cornerRadiusTopLeft=3, cornerRadiusTopRight=3)
    .encode(
        alt.X(
            "day_of_week:N",
            title="Day of Week",
            sort=day_order,
            axis=alt.Axis(labelFontSize=11, titleFontSize=13, labelAngle=0),
        ),
        alt.Y(
            "avg_price:Q",
            title="Avg Price ($)",
            scale=alt.Scale(zero=False),          # ← let Altair auto-scale, no hardcoded domain
            axis=alt.Axis(labelFontSize=11, titleFontSize=13, format="$.2f"),
        ),
        alt.Color("cab_type:N", scale=color_scale, legend=alt.Legend(title="Cab Type")),
        alt.XOffset("cab_type:N"),
        tooltip=[
            alt.Tooltip("day_of_week:N", title="Day"),
            alt.Tooltip("cab_type:N", title="Brand"),
            alt.Tooltip("avg_price:Q", title="Avg Price", format="$.2f"),
        ],
    )
)

text6 = (
    alt.Chart(daily)
    .mark_text(dy=-7, fontSize=9.5, fontWeight="bold")
    .encode(
        alt.X("day_of_week:N", sort=day_order),
        alt.Y("avg_price:Q", scale=alt.Scale(zero=False)),
        alt.XOffset("cab_type:N"),
        alt.Text("avg_price:Q", format="$.1f"),
        alt.Color("cab_type:N", scale=color_scale),
    )
)

chart6 = (
    (bars6 + text6)
    .properties(
        width=680,
        height=400,
        title=alt.TitleParams(
            "Average Ride Price by Day of Week",
            fontSize=16,
            fontWeight="bold",
            anchor="start",
            subtitle="Grouped by brand  •  Y-axis zoomed to highlight relative differences",
            subtitleFontSize=12,
            subtitleColor="#666",
        ),
    )
    .configure_view(strokeWidth=0)
    .configure_axis(grid=True, gridColor="#e8e8e8", gridOpacity=0.6)
    .configure_legend(labelFontSize=11, titleFontSize=12)
)
chart6

In [ ]:
surge_counts = (
    df.groupby(["surge_multiplier", "cab_type"])
    .size()
    .reset_index(name="count")
)

bars7 = (
    alt.Chart(surge_counts)
    .mark_bar(cornerRadiusTopLeft=3, cornerRadiusTopRight=3, opacity=0.85)
    .encode(
        alt.X(
            "surge_multiplier:O",
            title="Surge Multiplier",
            axis=alt.Axis(labelFontSize=11, titleFontSize=13, labelAngle=0),
        ),
        alt.Y(
            "count:Q",
            title="Number of Rides",
            axis=alt.Axis(labelFontSize=11, titleFontSize=13),
        ),
        alt.Color("cab_type:N", scale=color_scale, legend=alt.Legend(title="Cab Type")),
        alt.XOffset("cab_type:N"),
        tooltip=[
            alt.Tooltip("surge_multiplier:O", title="Surge"),
            alt.Tooltip("cab_type:N", title="Brand"),
            alt.Tooltip("count:Q", title="Rides", format=","),
        ],
    )
)

# Regression — treat surge_multiplier as Q for regression line
reg_uber7 = (
    alt.Chart(surge_counts[surge_counts["cab_type"] == "Uber"])
    .transform_regression("surge_multiplier", "count")
    .mark_line(strokeWidth=2, color="#1a1a2e", strokeDash=[6, 3], opacity=0.7)
    .encode(x="surge_multiplier:Q", y="count:Q")
)

reg_lyft7 = (
    alt.Chart(surge_counts[surge_counts["cab_type"] == "Lyft"])
    .transform_regression("surge_multiplier", "count")
    .mark_line(strokeWidth=2, color="#E91E8C", strokeDash=[6, 3], opacity=0.7)
    .encode(x="surge_multiplier:Q", y="count:Q")
)

chart7 = (
    (bars7 + reg_uber7 + reg_lyft7)
    .properties(
        width=650,
        height=400,
        title=alt.TitleParams(
            "Surge Multiplier Distribution",
            fontSize=16,
            fontWeight="bold",
            anchor="start",
            subtitle="Dashed lines = demand decay regression  •  ~96% of rides have no surge (1.0x)",
            subtitleFontSize=12,
            subtitleColor="#666",
        ),
    )
    .configure_view(strokeWidth=0)
    .configure_axis(grid=True, gridColor="#e8e8e8", gridOpacity=0.6)
    .configure_legend(labelFontSize=11, titleFontSize=12)
)
chart7

In [ ]:
box8 = (
    alt.Chart(df)
    .mark_boxplot(size=40, outliers=alt.MarkConfig(size=6, opacity=0.1))
    .encode(
        alt.X(
            "surge_multiplier:O",
            title="Surge Multiplier",
            axis=alt.Axis(labelFontSize=11, titleFontSize=13, labelAngle=0),
        ),
        alt.Y(
            "price:Q",
            title="Price ($)",
            scale=alt.Scale(zero=False),
            axis=alt.Axis(labelFontSize=11, titleFontSize=13),
        ),
        alt.Color("cab_type:N", scale=color_scale, legend=alt.Legend(title="Cab Type")),
        tooltip=[
            alt.Tooltip("surge_multiplier:O", title="Surge"),
            alt.Tooltip("cab_type:N", title="Brand"),
            alt.Tooltip("mean(price):Q", title="Avg Price", format="$.2f"),
            alt.Tooltip("median(price):Q", title="Median Price", format="$.2f"),
        ],
    )
)

mean_dots8 = (
    alt.Chart(df)
    .mark_point(shape="diamond", size=80, filled=True, color="white", strokeWidth=1.2)
    .encode(
        x=alt.X("surge_multiplier:O"),
        y=alt.Y("mean(price):Q"),
        stroke=alt.Stroke("cab_type:N", scale=color_scale),
        xOffset=alt.XOffset("cab_type:N"),
        tooltip=[
            alt.Tooltip("surge_multiplier:O", title="Surge"),
            alt.Tooltip("mean(price):Q", title="Mean Price", format="$.2f"),
        ],
    )
)

chart8 = (
    (box8 + mean_dots8)
    .properties(
        width=650,
        height=400,
        title=alt.TitleParams(
            "Price Distribution by Surge Multiplier",
            fontSize=16,
            fontWeight="bold",
            anchor="start",
            subtitle="Diamond = mean  •  Quantifies price impact of demand-based surge pricing",
            subtitleFontSize=12,
            subtitleColor="#666",
        ),
    )
    .configure_view(strokeWidth=0)
    .configure_axis(grid=True, gridColor="#e8e8e8", gridOpacity=0.6)
    .configure_legend(labelFontSize=11, titleFontSize=12)
)
chart8

In [ ]:
route_avg = (
    df.groupby(["source", "destination"])["price"]
    .mean()
    .reset_index()
    .rename(columns={"price": "avg_price"})
)

loc_order = [
    "North Station", "West End", "Haymarket Square", "North End",
    "Beacon Hill", "Financial District", "South Station",
    "Theatre District", "Back Bay", "Fenway",
    "Boston University", "Northeastern University"
]

heatmap9 = (
    alt.Chart(route_avg)
    .mark_rect()
    .encode(
        alt.X(
            "source:N",
            title="Origin",
            sort=loc_order,
            axis=alt.Axis(labelFontSize=10, titleFontSize=13, labelAngle=-35),
        ),
        alt.Y(
            "destination:N",
            title="Destination",
            sort=loc_order,
            axis=alt.Axis(labelFontSize=10, titleFontSize=13),
        ),
        alt.Color(
            "avg_price:Q",
            title="Avg Price ($)",
            scale=alt.Scale(scheme="blues"),
            legend=alt.Legend(titleFontSize=11, labelFontSize=10, format="$.1f"),
        ),
        tooltip=[
            alt.Tooltip("source:N", title="From"),
            alt.Tooltip("destination:N", title="To"),
            alt.Tooltip("avg_price:Q", title="Avg Price", format="$.2f"),
        ],
    )
)

# ── Fix: use alt.condition at top level for text color ────────────────────
text9 = (
    alt.Chart(route_avg)
    .mark_text(fontSize=8.5, fontWeight="bold")
    .encode(
        alt.X("source:N", sort=loc_order),
        alt.Y("destination:N", sort=loc_order),
        alt.Text("avg_price:Q", format="$.0f"),
        color=alt.condition(
            "datum.avg_price > 20",
            alt.value("white"),
            alt.value("#333333"),
        ),
    )
)

chart9 = (
    (heatmap9 + text9)
    .properties(
        width=560,
        height=520,
        title=alt.TitleParams(
            "Average Price by Route (Source → Destination)",
            fontSize=16,
            fontWeight="bold",
            anchor="start",
            subtitle="All brands combined  •  Darker = more expensive corridor",
            subtitleFontSize=12,
            subtitleColor="#666",
        ),
    )
    .configure_view(strokeWidth=0)
    .configure_axis(grid=False)
    .configure_legend(labelFontSize=11, titleFontSize=12)
)
chart9

In [ ]:
hourly_volume = (
    df.groupby(["hour", "cab_type"])
    .size()
    .reset_index(name="ride_count")
)

area10 = (
    alt.Chart(hourly_volume)
    .mark_area(opacity=0.4, interpolate="monotone")
    .encode(
        alt.X(
            "hour:Q",
            title="Hour of Day (0 = Midnight)",
            axis=alt.Axis(
                labelFontSize=11, titleFontSize=13,
                values=list(range(0, 24)),
                labelExpr="datum.value + ':00'",
            ),
        ),
        alt.Y(
            "ride_count:Q",
            title="Number of Rides",
            stack=None,
            axis=alt.Axis(labelFontSize=11, titleFontSize=13),
        ),
        alt.Color("cab_type:N", scale=color_scale, legend=alt.Legend(title="Cab Type")),
        tooltip=[
            alt.Tooltip("hour:Q", title="Hour"),
            alt.Tooltip("cab_type:N", title="Brand"),
            alt.Tooltip("ride_count:Q", title="Rides", format=","),
        ],
    )
)

line10 = (
    alt.Chart(hourly_volume)
    .mark_line(strokeWidth=2.5, interpolate="monotone")
    .encode(
        alt.X("hour:Q"),
        alt.Y("ride_count:Q", stack=None),
        alt.Color("cab_type:N", scale=color_scale),
    )
)

chart10 = (
    (area10 + line10)
    .properties(
        width=680,
        height=380,
        title=alt.TitleParams(
            "Ride Volume by Hour of Day",
            fontSize=16,
            fontWeight="bold",
            anchor="start",
            subtitle="Demand pattern across 24 hours  •  Overlapping areas = stacked transparency",
            subtitleFontSize=12,
            subtitleColor="#666",
        ),
    )
    .configure_view(strokeWidth=0)
    .configure_axis(grid=True, gridColor="#e8e8e8", gridOpacity=0.6)
    .configure_legend(labelFontSize=11, titleFontSize=12)
)

chart10

In [ ]:
scatter11 = (
    alt.Chart(df_sample)
    .mark_point(opacity=0.15, size=18)
    .encode(
        alt.X(
            "temperature:Q",
            title="Temperature (°F)",
            axis=alt.Axis(labelFontSize=11, titleFontSize=13),
        ),
        alt.Y(
            "price:Q",
            title="Price ($)",
            scale=alt.Scale(zero=False),
            axis=alt.Axis(labelFontSize=11, titleFontSize=13),
        ),
        alt.Color("cab_type:N", scale=color_scale, legend=alt.Legend(title="Cab Type")),
        tooltip=[
            alt.Tooltip("temperature:Q", title="Temp (°F)", format=".1f"),
            alt.Tooltip("price:Q", title="Price ($)", format="$.2f"),
            alt.Tooltip("cab_type:N", title="Brand"),
        ],
    )
)

reg_uber11 = (
    alt.Chart(df_sample[df_sample["cab_type"] == "Uber"])
    .transform_regression("temperature", "price")
    .mark_line(strokeWidth=2.5, color="#0e84b3", strokeDash=[6, 3])
    .encode(x="temperature:Q", y="price:Q")
)

reg_lyft11 = (
    alt.Chart(df_sample[df_sample["cab_type"] == "Lyft"])
    .transform_regression("temperature", "price")
    .mark_line(strokeWidth=2.5, color="#CA1521", strokeDash=[6, 3])
    .encode(x="temperature:Q", y="price:Q")
)

chart11 = (
    (scatter11 + reg_uber11 + reg_lyft11)
    .properties(
        width=680,
        height=400,
        title=alt.TitleParams(
            "Price vs Temperature",
            fontSize=16,
            fontWeight="bold",
            anchor="start",
            subtitle="30K sampled rides  •  Dashed lines = regression trend per brand",
            subtitleFontSize=12,
            subtitleColor="#666",
        ),
    )
    .configure_view(strokeWidth=0)
    .configure_axis(grid=True, gridColor="#e8e8e8", gridOpacity=0.6)
    .configure_legend(labelFontSize=11, titleFontSize=12)
)
chart11

In [ ]:
weather_avg = (
    df.groupby(["short_summary", "cab_type"])["price"]
    .mean()
    .reset_index()
    .rename(columns={"price": "avg_price"})
)

# Filter to conditions with enough data
condition_counts = df["short_summary"].value_counts()
valid_conditions = condition_counts[condition_counts > 500].index.tolist()
weather_avg = weather_avg[weather_avg["short_summary"].isin(valid_conditions)]

# Sort by avg price for cleaner visual
condition_order = (
    weather_avg.groupby("short_summary")["avg_price"]
    .mean()
    .sort_values(ascending=False)
    .index.tolist()
)

bars12 = (
    alt.Chart(weather_avg)
    .mark_bar(cornerRadiusTopLeft=3, cornerRadiusTopRight=3)
    .encode(
        alt.X(
            "short_summary:N",
            title="Weather Condition",
            sort=condition_order,
            axis=alt.Axis(labelFontSize=10, titleFontSize=13, labelAngle=-25),
        ),
        alt.Y(
            "avg_price:Q",
            title="Avg Price ($)",
            scale=alt.Scale(zero=False),
            axis=alt.Axis(labelFontSize=11, titleFontSize=13, format="$.2f"),
        ),
        alt.Color("cab_type:N", scale=color_scale, legend=alt.Legend(title="Cab Type")),
        alt.XOffset("cab_type:N"),
        tooltip=[
            alt.Tooltip("short_summary:N", title="Weather"),
            alt.Tooltip("cab_type:N", title="Brand"),
            alt.Tooltip("avg_price:Q", title="Avg Price", format="$.2f"),
        ],
    )
)

text12 = (
    alt.Chart(weather_avg)
    .mark_text(dy=-7, fontSize=9, fontWeight="bold")
    .encode(
        alt.X("short_summary:N", sort=condition_order),
        alt.Y("avg_price:Q", scale=alt.Scale(zero=False)),
        alt.XOffset("cab_type:N"),
        alt.Text("avg_price:Q", format="$.1f"),
        color=alt.condition(
            "datum.avg_price > 18",
            alt.value("#333333"),
            alt.value("#333333"),
        ),
    )
)

chart12 = (
    (bars12 + text12)
    .properties(
        width=700,
        height=420,
        title=alt.TitleParams(
            "Average Price by Weather Condition",
            fontSize=16,
            fontWeight="bold",
            anchor="start",
            subtitle="Conditions sorted by avg price  •  Only conditions with 500+ rides shown",
            subtitleFontSize=12,
            subtitleColor="#666",
        ),
    )
    .configure_view(strokeWidth=0)
    .configure_axis(grid=True, gridColor="#e8e8e8", gridOpacity=0.6)
    .configure_legend(labelFontSize=11, titleFontSize=12)
)
chart12

In [ ]:
route_brand = (
    df.groupby(["source", "destination", "cab_type"])["price"]
    .mean()
    .reset_index()
    .rename(columns={"price": "avg_price"})
)

# Create readable route label
route_brand["route"] = route_brand["source"] + " → " + route_brand["destination"]

# Keep only routes with both brands present
route_counts = route_brand.groupby("route")["cab_type"].nunique()
both_brands = route_counts[route_counts == 2].index.tolist()
route_brand = route_brand[route_brand["route"].isin(both_brands)]

# Sort routes by price difference (Lyft - Uber) for insight
pivot = route_brand.pivot_table(index="route", columns="cab_type", values="avg_price").reset_index()
pivot["diff"] = pivot["Lyft"] - pivot["Uber"]
route_order = pivot.sort_values("diff", ascending=False)["route"].tolist()

# Sample top 20 routes by absolute diff for readability
top_routes = pivot.reindex(pivot["diff"].abs().sort_values(ascending=False).index).head(20)["route"].tolist()
route_brand_top = route_brand[route_brand["route"].isin(top_routes)]

bars13 = (
    alt.Chart(route_brand_top)
    .mark_bar(cornerRadiusTopLeft=3, cornerRadiusTopRight=3)
    .encode(
        alt.Y(
            "route:N",
            title="Route",
            sort=top_routes,
            axis=alt.Axis(labelFontSize=9.5, titleFontSize=13),
        ),
        alt.X(
            "avg_price:Q",
            title="Avg Price ($)",
            scale=alt.Scale(zero=False),
            axis=alt.Axis(labelFontSize=11, titleFontSize=13, format="$.1f"),
        ),
        alt.Color("cab_type:N", scale=color_scale, legend=alt.Legend(title="Cab Type")),
        alt.YOffset("cab_type:N"),
        tooltip=[
            alt.Tooltip("route:N", title="Route"),
            alt.Tooltip("cab_type:N", title="Brand"),
            alt.Tooltip("avg_price:Q", title="Avg Price", format="$.2f"),
        ],
    )
)

chart13 = (
    bars13
    .properties(
        width=580,
        height=560,
        title=alt.TitleParams(
            "Uber vs Lyft — Price Comparison by Route",
            fontSize=16,
            fontWeight="bold",
            anchor="start",
            subtitle="Top 20 routes by largest price gap  •  Sorted by Lyft − Uber difference",
            subtitleFontSize=12,
            subtitleColor="#666",
        ),
    )
    .configure_view(strokeWidth=0)
    .configure_axis(grid=True, gridColor="#e8e8e8", gridOpacity=0.6)
    .configure_legend(labelFontSize=11, titleFontSize=12)
)
chart13

In [ ]:
iqr_stats = (
    df.groupby(["name", "cab_type"])["price"]
    .describe(percentiles=[0.25, 0.75])
    .reset_index()
    .rename(columns={"25%": "q1", "75%": "q3", "50%": "median"})
)
iqr_stats["iqr"]        = iqr_stats["q3"] - iqr_stats["q1"]
iqr_stats["lower_fence"] = iqr_stats["q1"] - 1.5 * iqr_stats["iqr"]
iqr_stats["upper_fence"] = iqr_stats["q3"] + 1.5 * iqr_stats["iqr"]

# Flag outliers
df2 = df.merge(
    iqr_stats[["name", "cab_type", "lower_fence", "upper_fence"]],
    on=["name", "cab_type"],
    how="left"
)
df2["is_outlier"] = (df2["price"] < df2["lower_fence"]) | (df2["price"] > df2["upper_fence"])

outlier_counts = (
    df2.groupby(["name", "cab_type"])
    .agg(
        total=("price", "count"),
        outliers=("is_outlier", "sum"),
    )
    .reset_index()
)
outlier_counts["outlier_pct"] = (outlier_counts["outliers"] / outlier_counts["total"] * 100).round(2)

print(outlier_counts[["name", "cab_type", "total", "outliers", "outlier_pct"]].to_string(index=False))

# Sort tiers by median price
tier_order = (
    df.groupby("name")["price"]
    .median()
    .sort_values()
    .index.tolist()
)

# ── Chart 15a: Boxplot per Tier ───────────────────────────────────────────

boxplot15 = (
    alt.Chart(df)
    .mark_boxplot(size=28, outliers=alt.MarkConfig(size=5, opacity=0.08))
    .encode(
        alt.X(
            "name:N",
            title="Ride Tier",
            sort=tier_order,
            axis=alt.Axis(labelFontSize=10, titleFontSize=13, labelAngle=-25),
        ),
        alt.Y(
            "price:Q",
            title="Price ($)",
            scale=alt.Scale(zero=False),
            axis=alt.Axis(labelFontSize=11, titleFontSize=13),
        ),
        alt.Color("cab_type:N", scale=color_scale, legend=alt.Legend(title="Cab Type")),
        tooltip=[
            alt.Tooltip("name:N", title="Tier"),
            alt.Tooltip("cab_type:N", title="Brand"),
            alt.Tooltip("mean(price):Q", title="Mean Price", format="$.2f"),
            alt.Tooltip("median(price):Q", title="Median Price", format="$.2f"),
        ],
    )
)

mean_dots15 = (
    alt.Chart(df)
    .mark_point(shape="diamond", size=70, filled=True, color="white", strokeWidth=1.2)
    .encode(
        x=alt.X("name:N", sort=tier_order),
        y=alt.Y("mean(price):Q"),
        stroke=alt.Stroke("cab_type:N", scale=color_scale),
        tooltip=[
            alt.Tooltip("name:N", title="Tier"),
            alt.Tooltip("mean(price):Q", title="Mean Price", format="$.2f"),
        ],
    )
)

boxchart = (boxplot15 + mean_dots15).properties(
    width=700,
    height=400,
    title=alt.TitleParams(
        "Price Outlier Analysis by Ride Tier",
        fontSize=16,
        fontWeight="bold",
        anchor="start",
        subtitle="Boxplot with IQR whiskers  •  Faint dots = outliers beyond 1.5×IQR  •  Diamond = mean",
        subtitleFontSize=12,
        subtitleColor="#666",
    ),
)

# ── Chart 15b: Outlier % per Tier (bar) ───────────────────────────────────

outl_bars = (
    alt.Chart(outlier_counts)
    .mark_bar(cornerRadiusTopLeft=3, cornerRadiusTopRight=3)
    .encode(
        alt.X(
            "name:N",
            title="Ride Tier",
            sort=tier_order,
            axis=alt.Axis(labelFontSize=10, titleFontSize=13, labelAngle=-25),
        ),
        alt.Y(
            "outlier_pct:Q",
            title="Outlier % of Rides",
            axis=alt.Axis(labelFontSize=11, titleFontSize=13, format=".1f"),
        ),
        alt.Color("cab_type:N", scale=color_scale, legend=alt.Legend(title="Cab Type")),
        alt.XOffset("cab_type:N"),
        tooltip=[
            alt.Tooltip("name:N", title="Tier"),
            alt.Tooltip("cab_type:N", title="Brand"),
            alt.Tooltip("outlier_pct:Q", title="Outlier %", format=".2f"),
            alt.Tooltip("outliers:Q", title="Outlier Count", format=","),
            alt.Tooltip("total:Q", title="Total Rides", format=","),
        ],
    )
)

outl_text = (
    alt.Chart(outlier_counts)
    .mark_text(dy=-7, fontSize=9, fontWeight="bold")
    .encode(
        alt.X("name:N", sort=tier_order),
        alt.Y("outlier_pct:Q"),
        alt.XOffset("cab_type:N"),
        alt.Text("outlier_pct:Q", format=".1f"),
        color=alt.value("#333333"),
    )
)

outlier_chart = (outl_bars + outl_text).properties(
    width=700,
    height=320,
    title=alt.TitleParams(
        "Outlier Rate (%) by Ride Tier",
        fontSize=15,
        fontWeight="bold",
        anchor="start",
        subtitle="Outliers defined as price beyond 1.5 × IQR fence",
        subtitleFontSize=11,
        subtitleColor="#666",
    ),
)

# ── Combine vertically ────────────────────────────────────────────────────

chart15 = (
    alt.vconcat(boxchart, outlier_chart)
    .configure_view(strokeWidth=0)
    .configure_axis(grid=True, gridColor="#e8e8e8", gridOpacity=0.6)
    .configure_legend(labelFontSize=11, titleFontSize=12)
)
chart15

Which platform is cheaper for the same route and conditions?

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Feature creation ──────────────────────────────────────────────────────────
df['route'] = df['source'] + ' → ' + df['destination']
df['price_per_mile'] = df['price'] / df['distance']

# Time-of-day buckets
def time_bucket(h):
    if   6 <= h < 10: return 'Morning Commute'
    elif 10 <= h < 16: return 'Daytime'
    elif 16 <= h < 20: return 'Evening Commute'
    elif 20 <= h < 24: return 'Nightlife'
    else:              return 'Late Night'

df['time_period'] = df['hour'].apply(time_bucket)

TIME_ORDER = ['Morning Commute', 'Daytime', 'Evening Commute', 'Nightlife', 'Late Night']
COLORS = {'Uber': '#1a1a2e', 'Lyft': '#E91E8C'}

print(f"Routes: {df['route'].nunique()}  |  Platforms: {df['cab_type'].unique()}")

In [ ]:
route_agg = (
    df.groupby(['route', 'cab_type'])
    .agg(
        median_price    = ('price',         'median'),
        median_ppm      = ('price_per_mile','median'),
        mean_price      = ('price',         'mean'),
        ride_count      = ('price',         'count'),
    )
    .reset_index()
)

# Pivot: one row per route, Uber and Lyft as columns
uber = (route_agg[route_agg.cab_type == 'Uber']
        .set_index('route')[['median_price','median_ppm','ride_count']]
        .add_prefix('uber_'))
lyft = (route_agg[route_agg.cab_type == 'Lyft']
        .set_index('route')[['median_price','median_ppm','ride_count']]
        .add_prefix('lyft_'))

pivot = uber.join(lyft).reset_index()

# Who is cheaper and by how much
pivot['price_diff']   = pivot['lyft_median_price'] - pivot['uber_median_price']   # + = Uber cheaper
pivot['ppm_diff']     = pivot['lyft_median_ppm']   - pivot['uber_median_ppm']
pivot['cheaper']      = pivot['price_diff'].apply(lambda x: 'Uber' if x > 0 else 'Lyft')
pivot['pct_cheaper']  = (pivot['price_diff'].abs() / pivot[['uber_median_price','lyft_median_price']].min(axis=1) * 100).round(1)

pivot.sort_values('price_diff', inplace=True)
pivot.head()

In [ ]:
fig = px.bar(
    pivot,
    x='price_diff',
    y='route',
    color='cheaper',
    color_discrete_map=COLORS,
    orientation='h',
    hover_data={
        'uber_median_price': ':.2f',
        'lyft_median_price': ':.2f',
        'pct_cheaper':       ':.1f',
        'cheaper':           True,
    },
    labels={
        'price_diff':         'Lyft − Uber median price ($)',
        'route':              'Route',
        'uber_median_price':  'Uber median ($)',
        'lyft_median_price':  'Lyft median ($)',
        'pct_cheaper':        '% cheaper',
    },
    title='<b>Q1 — Which platform is cheaper per route?</b><br>'
          '<sup>Positive bar → Uber cheaper &nbsp;|&nbsp; Negative bar → Lyft cheaper &nbsp;|&nbsp; Lyft − Uber median price</sup>',
)
fig.add_vline(x=0, line_width=1.5, line_color='grey')
fig.update_layout(
    height=2800,
    yaxis={'categoryorder': 'total ascending'},
    legend_title_text='Cheaper platform',
    margin=dict(l=220, r=40, t=80, b=40),
    font=dict(size=11),
)
fig.show()

In [ ]:
pivot[['src', 'dst']] = pivot['route'].str.split(' → ', expand=True)

heat_price = pivot.pivot(index='src', columns='dst', values='price_diff')

fig = px.imshow(
    heat_price,
    color_continuous_scale='RdBu',       # blue = Uber cheaper, red = Lyft cheaper
    color_continuous_midpoint=0,
    text_auto='.2f',
    aspect='auto',
    title='<b>Price Difference Heatmap: Lyft − Uber ($)</b><br>'
          '<sup>Blue = Uber cheaper &nbsp;|&nbsp; Red = Lyft cheaper</sup>',
    labels={'color': 'Lyft − Uber ($)', 'x': 'Destination', 'y': 'Source'},
)
fig.update_layout(height=600, margin=dict(t=80))
fig.update_xaxes(tickangle=30)
fig.show()

In [ ]:
time_route_agg = (
    df.groupby(['cab_type', 'time_period'])
    .agg(median_price=('price', 'median'), ride_count=('price', 'count'))
    .reset_index()
)

fig = px.bar(
    time_route_agg,
    x='time_period',
    y='median_price',
    color='cab_type',
    barmode='group',
    color_discrete_map=COLORS,
    category_orders={'time_period': TIME_ORDER},
    text_auto='.2f',
    hover_data={'ride_count': True},
    labels={
        'median_price': 'Median Price ($)',
        'time_period':  'Time Period',
        'cab_type':     'Platform',
        'ride_count':   'Rides',
    },
    title='<b>Q1 — Median Price by Platform and Time of Day</b><br>'
          '<sup>Controlling for when the ride was taken</sup>',
)
fig.update_layout(height=450, legend_title_text='Platform')
fig.update_traces(textposition='outside')
fig.show()

In [ ]:
uber_cheaper_n = (pivot['cheaper'] == 'Uber').sum()
lyft_cheaper_n = (pivot['cheaper'] == 'Lyft').sum()
total_routes   = len(pivot)

uber_overall = df[df.cab_type == 'Uber']['price'].median()
lyft_overall = df[df.cab_type == 'Lyft']['price'].median()

uber_ppm = df[df.cab_type == 'Uber']['price_per_mile'].median()
lyft_ppm = df[df.cab_type == 'Lyft']['price_per_mile'].median()

print("=" * 55)
print("Q1  SUMMARY — Platform Price Comparison")
print("=" * 55)
print(f"\nOverall median price:     Uber ${uber_overall:.2f}  |  Lyft ${lyft_overall:.2f}")
print(f"Overall median $/mile:    Uber ${uber_ppm:.2f}  |  Lyft ${lyft_ppm:.2f}")
print(f"\nRoutes where Uber is cheaper:  {uber_cheaper_n} / {total_routes}  ({uber_cheaper_n/total_routes*100:.0f}%)")
print(f"Routes where Lyft is cheaper:  {lyft_cheaper_n} / {total_routes}  ({lyft_cheaper_n/total_routes*100:.0f}%)")
print(f"\nBiggest Uber advantage: {pivot.iloc[-1]['route']}  (Lyft costs ${pivot.iloc[-1]['price_diff']:.2f} more)")
print(f"Biggest Lyft advantage: {pivot.iloc[0]['route']}   (Uber costs ${abs(pivot.iloc[0]['price_diff']):.2f} more)")

How much is distance vs surge vs price

In [ ]:
import plotly.graph_objects as go
import numpy as np
from sklearn.linear_model import LinearRegression

# ── Sample for render performance ─────────────────────────────────────────────
df_3d = df.sample(6000, random_state=42)

# ── Fit OLS on the two continuous predictors ──────────────────────────────────
X_reg = df[['distance', 'surge_multiplier']].values
y_reg = df['price'].values
model = LinearRegression().fit(X_reg, y_reg)

coef_dist  = model.coef_[0]
coef_surge = model.coef_[1]
intercept  = model.intercept_

# ── Build regression plane (meshgrid) ─────────────────────────────────────────
d_vals = np.linspace(df['distance'].quantile(0.01), df['distance'].quantile(0.99), 50)
s_vals = np.linspace(df['surge_multiplier'].min(),  df['surge_multiplier'].max(),  20)
D, S   = np.meshgrid(d_vals, s_vals)
Z_plane = model.predict(np.c_[D.ravel(), S.ravel()]).reshape(D.shape)

# ── Product type colour map ───────────────────────────────────────────────────
PRODUCT_COLORS = {
    'UberPool':    '#b5c8e2',
    'UberX':       '#6baed6',
    'UberXL':      '#2171b5',
    'Black':       '#08306b',
    'Black SUV':   '#041d40',
    'WAV':         '#737373',
    'Shared':      '#f9b8d4',
    'Lyft':        '#e878ac',
    'Lyft XL':     '#E91E8C',
    'Lux':         '#b5006e',
    'Lux Black':   '#7a004b',
    'Lux Black XL':'#3d0025',
}

fig = go.Figure()

# Regression plane
fig.add_trace(go.Surface(
    x=D, y=S, z=Z_plane,
    opacity=0.35,
    colorscale=[[0, '#c6dbef'], [1, '#08306b']],
    showscale=False,
    name='Regression plane',
    hovertemplate='Distance: %{x:.1f} mi<br>Surge: %{y:.2f}x<br>Predicted: $%{z:.2f}<extra>Regression plane</extra>',
))

# Scatter — one trace per product type
for product, color in PRODUCT_COLORS.items():
    sub = df_3d[df_3d['name'] == product]
    if sub.empty:
        continue
    fig.add_trace(go.Scatter3d(
        x=sub['distance'],
        y=sub['surge_multiplier'],
        z=sub['price'],
        mode='markers',
        name=product,
        marker=dict(size=2, color=color, opacity=0.45),
        hovertemplate=(
            f'<b>{product}</b><br>'
            'Distance: %{x:.2f} mi<br>'
            'Surge: %{y:.2f}x<br>'
            'Price: $%{z:.2f}<extra></extra>'
        ),
    ))

fig.update_layout(
    title=dict(
        text=(
            '<b>Q3 — Price as a function of Distance, Surge & Product Type</b><br>'
            f'<sup>Regression plane: Price = {intercept:.2f} '
            f'+ {coef_dist:.2f}×Distance '
            f'+ {coef_surge:.2f}×Surge &nbsp;|&nbsp; '
            f'Points coloured by product type</sup>'
        ),
        font=dict(size=14),
    ),
    scene=dict(
        xaxis=dict(title='Distance (miles)', backgroundcolor='#f8f8f8', gridcolor='#ddd'),
        yaxis=dict(title='Surge Multiplier',  backgroundcolor='#f8f8f8', gridcolor='#ddd'),
        zaxis=dict(title='Price ($)',          backgroundcolor='#f8f8f8', gridcolor='#ddd'),
        camera=dict(eye=dict(x=1.7, y=-1.7, z=0.8)),
    ),
    legend=dict(title='Product type', itemsizing='constant'),
    height=750,
    margin=dict(t=100, b=20, l=20, r=20),
)

fig.show()

# ── Print weights ─────────────────────────────────────────────────────────────
print("=" * 45)
print("OLS Regression Weights (continuous features)")
print("=" * 45)
print(f"  Intercept:         ${intercept:.2f}")
print(f"  Distance ($/mile): ${coef_dist:.2f}  ← per additional mile")
print(f"  Surge ($/1x):      ${coef_surge:.2f}  ← per 1x surge increase")
print(f"\n  R² (dist + surge only): {model.score(X_reg, y_reg)*100:.1f}%")

In [ ]:
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import numpy as np
import pandas as pd

# ── Feature set for clustering ────────────────────────────────────────────────
CLUSTER_FEATURES = [
    'hour', 'surge_multiplier', 'price',
    'distance', 'temperature', 'precipProbability', 'windSpeed'
]

if 'day_of_week' not in df.columns:
    df['day_of_week'] = pd.to_datetime(df['datetime']).dt.day_name()
df['is_weekend'] = df['day_of_week'].isin(['Saturday', 'Sunday']).astype(int)
CLUSTER_FEATURES.append('is_weekend')

df_c = df[CLUSTER_FEATURES + ['cab_type', 'name', 'day_of_week']].dropna().copy()

# ── Scale ─────────────────────────────────────────────────────────────────────
scaler  = StandardScaler()
X_scaled = scaler.fit_transform(df_c[CLUSTER_FEATURES])

# ── Elbow to pick k ───────────────────────────────────────────────────────────
inertias = []
K_RANGE  = range(2, 10)
for k in K_RANGE:
    inertias.append(KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_scaled).inertia_)

fig_elbow = go.Figure(go.Scatter(
    x=list(K_RANGE), y=inertias,
    mode='lines+markers',
    marker=dict(size=8, color='#4C78A8'),
    line=dict(width=2),
))
fig_elbow.update_layout(
    title='<b>Elbow Curve — choose k</b>',
    xaxis_title='Number of clusters (k)',
    yaxis_title='Inertia',
    height=350, width=550,
    plot_bgcolor='white',
)
fig_elbow.update_xaxes(showgrid=True, gridcolor='#eee', dtick=1)
fig_elbow.update_yaxes(showgrid=True, gridcolor='#eee')
fig_elbow.show()

# ── Fit KMeans with k=6 ───────────────────────────────────────────────────────
K = 6
km = KMeans(n_clusters=K, random_state=42, n_init=10)
df_c['cluster'] = km.fit_predict(X_scaled)

# ── Profile each cluster (raw means) ─────────────────────────────────────────
profile_raw  = df_c.groupby('cluster')[CLUSTER_FEATURES].mean()

# Auto-label based on dominant raw feature values
def auto_label(row):
    if row['surge_multiplier'] >= 1.4:
        return 'Surge Heavy'
    elif row['hour'] >= 20 or row['hour'] <= 3:
        return 'Nightlife'
    elif 6 <= row['hour'] <= 9 and row['is_weekend'] < 0.3:
        return 'Morning Commute'
    elif 16 <= row['hour'] <= 19 and row['is_weekend'] < 0.3:
        return 'Evening Commute'
    elif row['precipProbability'] >= 0.4 or row['temperature'] < 38:
        return 'Bad Weather'
    else:
        return 'Standard'

profile_raw['regime'] = profile_raw.apply(auto_label, axis=1)
cluster_to_label = profile_raw['regime'].to_dict()
df_c['regime'] = df_c['cluster'].map(cluster_to_label)

REGIME_COLORS = {
    'Morning Commute': '#F58518',
    'Evening Commute': '#E45756',
    'Nightlife':       '#7B2D8B',
    'Surge Heavy':     '#E91E8C',
    'Bad Weather':     '#4C78A8',
    'Standard':        '#54A24B',
}

# ── Normalize profile for heatmap ─────────────────────────────────────────────
profile_norm = profile_raw[CLUSTER_FEATURES].copy()
profile_norm = (profile_norm - profile_norm.min()) / (profile_norm.max() - profile_norm.min())
profile_norm.index = profile_norm.index.map(cluster_to_label)

FEATURE_LABELS = {
    'hour':              'Hour of day',
    'surge_multiplier':  'Surge multiplier',
    'price':             'Price',
    'distance':          'Distance',
    'temperature':       'Temperature',
    'precipProbability': 'Precip probability',
    'windSpeed':         'Wind speed',
    'is_weekend':        'Is weekend',
}
profile_norm.rename(columns=FEATURE_LABELS, inplace=True)

# ── Regime size + avg price ───────────────────────────────────────────────────
regime_stats = (
    df_c.groupby('regime')
    .agg(count=('price', 'count'), avg_price=('price', 'mean'), avg_surge=('surge_multiplier', 'mean'))
    .reset_index()
    .sort_values('avg_price', ascending=False)
)

# ── Hour distribution per regime ──────────────────────────────────────────────
hour_dist = (
    df_c.groupby(['hour', 'regime'])
    .size()
    .reset_index(name='count')
)

# ── Main figure ───────────────────────────────────────────────────────────────
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Regime profiles (normalised feature means)',
        'Avg price & ride volume per regime',
        'Hour-of-day distribution by regime',
        'Surge multiplier distribution by regime',
    ),
    vertical_spacing=0.16,
    horizontal_spacing=0.1,
    row_heights=[0.48, 0.52],
    specs=[[{'colspan': 2}, None],
           [{},              {}]],
)

# Row 1: Profile heatmap (full width)
fig.add_trace(go.Heatmap(
    z=profile_norm.values,
    x=profile_norm.columns.tolist(),
    y=profile_norm.index.tolist(),
    colorscale='RdBu_r',
    zmid=0.5,
    text=np.round(profile_norm.values, 2),
    texttemplate='%{text}',
    showscale=True,
    colorbar=dict(title='Normalised<br>mean', len=0.45, y=0.78),
    hovertemplate='Regime: %{y}<br>Feature: %{x}<br>Normalised value: %{z:.2f}<extra></extra>',
), row=1, col=1)

# Row 2 left: avg price bars + ride count as bubble size
for _, row_s in regime_stats.iterrows():
    fig.add_trace(go.Bar(
        x=[row_s['regime']],
        y=[row_s['avg_price']],
        name=row_s['regime'],
        marker_color=REGIME_COLORS.get(row_s['regime'], '#888'),
        text=f"${row_s['avg_price']:.2f}<br>{row_s['count']//1000}k rides",
        textposition='outside',
        showlegend=True,
    ), row=2, col=1)

# Row 2 right: hour distribution lines
for regime in df_c['regime'].unique():
    sub = hour_dist[hour_dist['regime'] == regime]
    fig.add_trace(go.Scatter(
        x=sub['hour'],
        y=sub['count'],
        mode='lines',
        name=regime,
        line=dict(color=REGIME_COLORS.get(regime, '#888'), width=2.5),
        showlegend=False,
        hovertemplate=f'<b>{regime}</b><br>Hour: %{{x}}:00<br>Rides: %{{y:,}}<extra></extra>',
    ), row=2, col=2)

fig.update_layout(
    title=dict(
        text='<b>Q6 — Pricing Regimes: KMeans Clustering (k=6)</b><br>'
             '<sup>Features: hour, weekend, surge, price, distance, temperature, precipitation, wind</sup>',
        font=dict(size=15),
    ),
    height=820,
    margin=dict(t=100, b=60, l=60, r=60),
    plot_bgcolor='white',
    paper_bgcolor='white',
    barmode='group',
    legend=dict(title='Regime', x=1.01, y=0.35),
)

fig.update_xaxes(showgrid=True, gridcolor='#eee')
fig.update_yaxes(showgrid=True, gridcolor='#eee')
fig.update_xaxes(title_text='Hour of day (0 = midnight)', row=2, col=2)
fig.update_yaxes(title_text='Avg Price ($)', row=2, col=1)
fig.update_yaxes(title_text='Ride count', row=2, col=2)
fig.update_xaxes(tickangle=20, row=2, col=1)

fig.show()

# ── Summary table ─────────────────────────────────────────────────────────────
print("\nRegime Summary:")
print(regime_stats.to_string(index=False))

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ── Features ──────────────────────────────────────────────────────────────────
if 'day_of_week' not in df.columns:
    df['day_of_week'] = pd.to_datetime(df['datetime']).dt.day_name()
if 'is_weekend' not in df.columns:
    df['is_weekend'] = df['day_of_week'].isin(['Saturday', 'Sunday']).astype(int)
if 'price_per_mile' not in df.columns:
    df['price_per_mile'] = df['price'] / df['distance']

NUM_FEATURES = ['distance', 'surge_multiplier', 'hour', 'is_weekend',
                'temperature', 'precipProbability', 'windSpeed']
CAT_FEATURES = ['name', 'cab_type', 'source', 'destination']
TARGET       = 'price'

df_model = df[NUM_FEATURES + CAT_FEATURES + [TARGET]].dropna().copy()

X = df_model[NUM_FEATURES + CAT_FEATURES]
y = df_model[TARGET]

# ── Train / test split ────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train: {len(X_train):,}  |  Test: {len(X_test):,}")

# ── Preprocessing + model pipeline ───────────────────────────────────────────
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(),                              NUM_FEATURES),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CAT_FEATURES),
])

pipe = Pipeline([
    ('prep',  preprocessor),
    ('model', LinearRegression()),
])

pipe.fit(X_train, y_train)

# ── Predictions & metrics ─────────────────────────────────────────────────────
y_pred_train = pipe.predict(X_train)
y_pred_test  = pipe.predict(X_test)

metrics = {
    'MAE':  {'train': mean_absolute_error(y_train, y_pred_train),
             'test':  mean_absolute_error(y_test,  y_pred_test)},
    'RMSE': {'train': mean_squared_error(y_train, y_pred_train) ** 0.5,
             'test':  mean_squared_error(y_test,  y_pred_test)  ** 0.5},
    'R²':   {'train': r2_score(y_train, y_pred_train),
             'test':  r2_score(y_test,  y_pred_test)},
}

print("\n{:<6} {:>12} {:>12}".format('Metric', 'Train', 'Test'))
print("-" * 32)
for m, vals in metrics.items():
    fmt = '.4f' if m == 'R²' else '.2f'
    print(f"{m:<6} {vals['train']:>12{fmt}} {vals['test']:>12{fmt}}")

# ── Coefficient table — top drivers ──────────────────────────────────────────
ohe_names  = pipe['prep'].named_transformers_['cat'].get_feature_names_out(CAT_FEATURES)
all_names  = np.array(NUM_FEATURES + list(ohe_names))
coefs      = pipe['model'].coef_

coef_df = (pd.DataFrame({'feature': all_names, 'coefficient': coefs})
             .reindex(pd.Series(coefs).abs().sort_values(ascending=False).index)
             .head(20)
             .reset_index(drop=True))

# ── Residuals ─────────────────────────────────────────────────────────────────
residuals  = y_test.values - y_pred_test
sample_idx = np.random.default_rng(42).integers(0, len(y_test), 4000)

# ── Plots ─────────────────────────────────────────────────────────────────────
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Top 20 feature coefficients',
        'Actual vs Predicted (test set)',
        'Residuals distribution',
        'Residuals vs Fitted',
    ),
    vertical_spacing=0.16,
    horizontal_spacing=0.12,
)

# Top coefficients
colors = ['#E45756' if c < 0 else '#4C78A8' for c in coef_df['coefficient']]
fig.add_trace(go.Bar(
    x=coef_df['coefficient'],
    y=coef_df['feature'],
    orientation='h',
    marker_color=colors,
    showlegend=False,
    hovertemplate='%{y}: %{x:.3f}<extra></extra>',
), row=1, col=1)
fig.add_vline(x=0, line_width=1, line_color='grey', row=1, col=1)

# Actual vs Predicted scatter
fig.add_trace(go.Scatter(
    x=y_test.values[sample_idx],
    y=y_pred_test[sample_idx],
    mode='markers',
    marker=dict(size=3, color='#4C78A8', opacity=0.35),
    showlegend=False,
    hovertemplate='Actual: $%{x:.2f}<br>Predicted: $%{y:.2f}<extra></extra>',
), row=1, col=2)
# Perfect prediction line
lim = [0, y_test.max()]
fig.add_trace(go.Scatter(
    x=lim, y=lim,
    mode='lines',
    line=dict(color='#E45756', dash='dash', width=1.5),
    showlegend=False,
), row=1, col=2)

# Residuals histogram
fig.add_trace(go.Histogram(
    x=residuals,
    nbinsx=80,
    marker_color='#4C78A8',
    opacity=0.75,
    showlegend=False,
    hovertemplate='Residual: %{x:.2f}<br>Count: %{y}<extra></extra>',
), row=2, col=1)
fig.add_vline(x=0, line_width=1.5, line_dash='dash', line_color='#E45756', row=2, col=1)

# Residuals vs Fitted
fig.add_trace(go.Scatter(
    x=y_pred_test[sample_idx],
    y=residuals[sample_idx],
    mode='markers',
    marker=dict(size=3, color='#4C78A8', opacity=0.3),
    showlegend=False,
    hovertemplate='Fitted: $%{x:.2f}<br>Residual: $%{y:.2f}<extra></extra>',
), row=2, col=2)
fig.add_hline(y=0, line_width=1.5, line_dash='dash', line_color='#E45756', row=2, col=2)

fig.update_layout(
    title=dict(
        text=(
            f'<b>Linear Regression Baseline</b><br>'
            f'<sup>Test R²: {metrics["R²"]["test"]:.4f} &nbsp;|&nbsp;'
            f' MAE: ${metrics["MAE"]["test"]:.2f} &nbsp;|&nbsp;'
            f' RMSE: ${metrics["RMSE"]["test"]:.2f}</sup>'
        ),
        font=dict(size=15),
    ),
    height=820,
    margin=dict(t=100, b=60, l=60, r=40),
    plot_bgcolor='white',
    paper_bgcolor='white',
)

fig.update_xaxes(showgrid=True, gridcolor='#eee')
fig.update_yaxes(showgrid=True, gridcolor='#eee')
fig.update_xaxes(title_text='Coefficient value',  row=1, col=1)
fig.update_xaxes(title_text='Actual price ($)',    row=1, col=2)
fig.update_yaxes(title_text='Predicted price ($)', row=1, col=2)
fig.update_xaxes(title_text='Residual ($)',        row=2, col=1)
fig.update_xaxes(title_text='Fitted value ($)',    row=2, col=2)
fig.update_yaxes(title_text='Residual ($)',        row=2, col=2)

fig.show()

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ── Features (same as linear baseline) ───────────────────────────────────────
if 'day_of_week' not in df.columns:
    df['day_of_week'] = pd.to_datetime(df['datetime']).dt.day_name()
if 'is_weekend' not in df.columns:
    df['is_weekend'] = df['day_of_week'].isin(['Saturday', 'Sunday']).astype(int)

NUM_FEATURES = ['distance', 'surge_multiplier', 'hour', 'is_weekend',
                'temperature', 'precipProbability', 'windSpeed']
CAT_FEATURES = ['name', 'cab_type', 'source', 'destination']
TARGET       = 'price'
DEGREE       = 3   # change to 3 for cubic (much slower, risks overfitting)

df_model = df[NUM_FEATURES + CAT_FEATURES + [TARGET]].dropna().copy()
X = df_model[NUM_FEATURES + CAT_FEATURES]
y = df_model[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train: {len(X_train):,}  |  Test: {len(X_test):,}")

# ── Baseline linear pipeline (for comparison) ─────────────────────────────────
base_preprocessor = ColumnTransformer([
    ('num', StandardScaler(),                                              NUM_FEATURES),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False),   CAT_FEATURES),
])
base_pipe = Pipeline([('prep', base_preprocessor), ('model', LinearRegression())])
base_pipe.fit(X_train, y_train)
y_pred_base = base_pipe.predict(X_test)

# ── Polynomial pipeline ───────────────────────────────────────────────────────
# PolynomialFeatures only on numerics — applying to OHE dummies is meaningless
num_poly_pipe = Pipeline([
    ('poly',  PolynomialFeatures(degree=DEGREE, include_bias=False)),
    ('scale', StandardScaler()),
])

poly_preprocessor = ColumnTransformer([
    ('num', num_poly_pipe,                                                 NUM_FEATURES),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False),   CAT_FEATURES),
])

poly_pipe = Pipeline([('prep', poly_preprocessor), ('model', LinearRegression())])
poly_pipe.fit(X_train, y_train)

n_poly_features = poly_pipe['prep'].named_transformers_['num']['poly'].n_output_features_
print(f"\nDegree-{DEGREE} polynomial expanded {len(NUM_FEATURES)} numeric features → {n_poly_features} features")

# ── Predictions & metrics ─────────────────────────────────────────────────────
y_pred_train = poly_pipe.predict(X_train)
y_pred_test  = poly_pipe.predict(X_test)

def metrics(y_true, y_pred):
    return {
        'MAE':  mean_absolute_error(y_true, y_pred),
        'RMSE': mean_squared_error(y_true, y_pred) ** 0.5,
        'R²':   r2_score(y_true, y_pred),
    }

m_train = metrics(y_train, y_pred_train)
m_test  = metrics(y_test,  y_pred_test)
m_base  = metrics(y_test,  y_pred_base)

print(f"\n{'Metric':<6} {'Linear (test)':>15} {'Poly train':>12} {'Poly test':>12} {'Gain':>10}")
print("-" * 58)
for m in ['MAE', 'RMSE', 'R²']:
    fmt = '.4f' if m == 'R²' else '.2f'
    gain = m_test[m] - m_base[m]
    sign = '+' if gain >= 0 else ''
    print(f"{m:<6} {m_base[m]:>15{fmt}} {m_train[m]:>12{fmt}} {m_test[m]:>12{fmt}} {sign}{gain:>9{fmt}}")

# ── Top coefficients ──────────────────────────────────────────────────────────
poly_feat_names = (poly_pipe['prep']
                   .named_transformers_['num']['poly']
                   .get_feature_names_out(NUM_FEATURES))
ohe_names       = (poly_pipe['prep']
                   .named_transformers_['cat']
                   .get_feature_names_out(CAT_FEATURES))
all_names       = np.array(list(poly_feat_names) + list(ohe_names))
coefs           = poly_pipe['model'].coef_

coef_df = (pd.DataFrame({'feature': all_names, 'coefficient': coefs})
             .assign(abs_coef=lambda d: d['coefficient'].abs())
             .sort_values('abs_coef', ascending=False)
             .head(20)
             .reset_index(drop=True))

# ── Residuals ─────────────────────────────────────────────────────────────────
residuals  = y_test.values - y_pred_test
base_resid = y_test.values - y_pred_base
rng        = np.random.default_rng(42)
sample_idx = rng.integers(0, len(y_test), 4000)

# ── Plots ─────────────────────────────────────────────────────────────────────
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        f'Top 20 coefficients (degree-{DEGREE} polynomial features)',
        'Actual vs Predicted (test set)',
        'Residuals: Linear vs Polynomial',
        'Residuals vs Fitted',
    ),
    vertical_spacing=0.16,
    horizontal_spacing=0.12,
)

# Top coefficients
colors = ['#E45756' if c < 0 else '#4C78A8' for c in coef_df['coefficient']]
fig.add_trace(go.Bar(
    x=coef_df['coefficient'],
    y=coef_df['feature'],
    orientation='h',
    marker_color=colors,
    showlegend=False,
    hovertemplate='%{y}: %{x:.4f}<extra></extra>',
), row=1, col=1)
fig.add_vline(x=0, line_width=1, line_color='grey', row=1, col=1)

# Actual vs Predicted
fig.add_trace(go.Scatter(
    x=y_test.values[sample_idx],
    y=y_pred_test[sample_idx],
    mode='markers',
    marker=dict(size=3, color='#4C78A8', opacity=0.35),
    name='Polynomial',
    showlegend=True,
    hovertemplate='Actual: $%{x:.2f}<br>Predicted: $%{y:.2f}<extra></extra>',
), row=1, col=2)
lim = [0, float(y_test.max())]
fig.add_trace(go.Scatter(
    x=lim, y=lim,
    mode='lines',
    line=dict(color='#E45756', dash='dash', width=1.5),
    name='Perfect fit',
    showlegend=True,
), row=1, col=2)

# Residual overlay — linear vs polynomial
for resid, label, color in [
    (base_resid, 'Linear',     '#F58518'),
    (residuals,  'Polynomial', '#4C78A8'),
]:
    fig.add_trace(go.Histogram(
        x=resid,
        nbinsx=100,
        name=label,
        marker_color=color,
        opacity=0.55,
        histnorm='probability density',
        hovertemplate=f'{label} residual: %{{x:.2f}}<extra></extra>',
    ), row=2, col=1)
fig.add_vline(x=0, line_width=1.5, line_dash='dash', line_color='#E45756', row=2, col=1)

# Residuals vs Fitted
fig.add_trace(go.Scatter(
    x=y_pred_test[sample_idx],
    y=residuals[sample_idx],
    mode='markers',
    marker=dict(size=3, color='#4C78A8', opacity=0.3),
    showlegend=False,
    hovertemplate='Fitted: $%{x:.2f}<br>Residual: $%{y:.2f}<extra></extra>',
), row=2, col=2)
fig.add_hline(y=0, line_width=1.5, line_dash='dash', line_color='#E45756', row=2, col=2)

fig.update_layout(
    title=dict(
        text=(
            f'<b>Polynomial Regression (degree={DEGREE})</b><br>'
            f'<sup>'
            f'Test R²: {m_test["R²"]:.4f} &nbsp;|&nbsp;'
            f'MAE: ${m_test["MAE"]:.2f} &nbsp;|&nbsp;'
            f'RMSE: ${m_test["RMSE"]:.2f} &nbsp;|&nbsp;'
            f'vs Linear → R² {("+" if m_test["R²"]-m_base["R²"] >= 0 else "")}'
            f'{(m_test["R²"]-m_base["R²"]):.4f}'
            f'</sup>'
        ),
        font=dict(size=15),
    ),
    height=820,
    barmode='overlay',
    legend=dict(x=0.52, y=0.42),
    margin=dict(t=100, b=60, l=60, r=40),
    plot_bgcolor='white',
    paper_bgcolor='white',
)

fig.update_xaxes(showgrid=True, gridcolor='#eee')
fig.update_yaxes(showgrid=True, gridcolor='#eee')
fig.update_xaxes(title_text='Coefficient value',     row=1, col=1)
fig.update_xaxes(title_text='Actual price ($)',       row=1, col=2)
fig.update_yaxes(title_text='Predicted price ($)',    row=1, col=2)
fig.update_xaxes(title_text='Residual ($)',           row=2, col=1)
fig.update_yaxes(title_text='Probability density',    row=2, col=1)
fig.update_xaxes(title_text='Fitted value ($)',       row=2, col=2)
fig.update_yaxes(title_text='Residual ($)',           row=2, col=2)

fig.show()

In [ ]:

# ── Derived columns ───────────────────────────────────────────────────────────
if 'day_of_week' not in df.columns:
    df['day_of_week'] = pd.to_datetime(df['datetime']).dt.day_name()
if 'is_weekend' not in df.columns:
    df['is_weekend'] = df['day_of_week'].isin(['Saturday', 'Sunday']).astype(int)

def time_bucket(h):
    if   6 <= h < 10: return 'Morning Commute'
    elif 10 <= h < 16: return 'Daytime'
    elif 16 <= h < 20: return 'Evening Commute'
    elif 20 <= h < 24: return 'Nightlife'
    else:              return 'Late Night'

if 'time_period' not in df.columns:
    df['time_period'] = df['hour'].apply(time_bucket)

# ── Group rare weather conditions ─────────────────────────────────────────────
MIN_WEATHER_COUNT = 1000
weather_counts  = df['short_summary'].value_counts()
common_weather  = weather_counts[weather_counts >= MIN_WEATHER_COUNT].index
df['weather_group'] = df['short_summary'].where(df['short_summary'].isin(common_weather), 'Other')

# ── Combined stratification key ───────────────────────────────────────────────
df['strat_key'] = (
    df['weather_group'] + '__' +
    df['cab_type']      + '__' +
    df['time_period']
)

# Drop strat combinations with < 20 samples (can't safely stratify on them)
key_counts  = df['strat_key'].value_counts()
valid_keys  = key_counts[key_counts >= 20].index
df_model    = df[df['strat_key'].isin(valid_keys)].copy()

print(f"Unique strat combinations : {df_model['strat_key'].nunique()}")
print(f"Rows after rare-key drop  : {len(df_model):,}  "
      f"({len(df_model)/len(df)*100:.1f}% of full dataset)")

# ── Features ──────────────────────────────────────────────────────────────────
NUM_FEATURES = ['distance', 'surge_multiplier', 'hour', 'is_weekend',
                'temperature', 'precipProbability', 'windSpeed']
CAT_FEATURES = ['name', 'cab_type', 'source', 'destination']
TARGET       = 'price'

df_model = df_model[NUM_FEATURES + CAT_FEATURES + [TARGET,
           'strat_key', 'weather_group', 'time_period']].dropna()

X = df_model[NUM_FEATURES + CAT_FEATURES]
y = df_model[TARGET]

# ── Stratified split ──────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, df_model.index,
    test_size=0.2,
    random_state=42,
    stratify=df_model['strat_key'],
)
print(f"\nTrain: {len(X_train):,}  |  Test: {len(X_test):,}")

meta_test = df_model.loc[idx_test, ['weather_group', 'cab_type', 'time_period']].copy()

# ── Verify stratification ─────────────────────────────────────────────────────
def split_dist(col):
    tr = df_model.loc[idx_train, col].value_counts(normalize=True).rename('Train')
    te = df_model.loc[idx_test,  col].value_counts(normalize=True).rename('Test')
    return pd.concat([tr, te], axis=1).reset_index().rename(columns={'index': col})

dist_weather = split_dist('weather_group')
dist_cab     = split_dist('cab_type')
dist_time    = split_dist('time_period')

TIME_ORDER = ['Morning Commute','Daytime','Evening Commute','Nightlife','Late Night']

fig_strat = make_subplots(
    rows=1, cols=3,
    subplot_titles=('Weather condition', 'Cab type', 'Time of day'),
    horizontal_spacing=0.1,
)

for col, dist, order, ci in [
    ('weather_group', dist_weather, None,       1),
    ('cab_type',      dist_cab,     None,       2),
    ('time_period',   dist_time,    TIME_ORDER, 3),
]:
    if order:
        dist = dist.set_index(col).reindex(order).reset_index()
    for split, color in [('Train', '#4C78A8'), ('Test', '#E45756')]:
        fig_strat.add_trace(go.Bar(
            x=dist[col],
            y=dist[split],
            name=split,
            marker_color=color,
            opacity=0.75,
            showlegend=(ci == 1),
            hovertemplate=f'{split}: %{{y:.2%}}<extra></extra>',
        ), row=1, col=ci)

fig_strat.update_layout(
    title='<b>Stratification check — Train vs Test distributions</b><br>'
          '<sup>Bars should be near-identical if stratification worked</sup>',
    barmode='group',
    height=380,
    margin=dict(t=90, b=60),
    plot_bgcolor='white',
    paper_bgcolor='white',
    legend=dict(title='Split'),
)
fig_strat.update_yaxes(tickformat='.0%', showgrid=True, gridcolor='#eee')
fig_strat.update_xaxes(tickangle=20)
fig_strat.show()

# ── Fit model ─────────────────────────────────────────────────────────────────
preprocessor = ColumnTransformer([
    ('num', StandardScaler(),                                             NUM_FEATURES),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False),  CAT_FEATURES),
])
pipe = Pipeline([('prep', preprocessor), ('model', LinearRegression())])
pipe.fit(X_train, y_train)

y_pred_train = pipe.predict(X_train)
y_pred_test  = pipe.predict(X_test)

def metrics(y_true, y_pred):
    return dict(
        MAE  = mean_absolute_error(y_true, y_pred),
        RMSE = mean_squared_error(y_true,  y_pred) ** 0.5,
        R2   = r2_score(y_true,            y_pred),
    )

m_train = metrics(y_train, y_pred_train)
m_test  = metrics(y_test,  y_pred_test)

print(f"\n{'Metric':<6} {'Train':>12} {'Test':>12}")
print("-" * 32)
for m in ['MAE', 'RMSE', 'R2']:
    fmt = '.4f' if m == 'R2' else '.2f'
    print(f"{m:<6} {m_train[m]:>12{fmt}} {m_test[m]:>12{fmt}}")

# ── Top coefficients ──────────────────────────────────────────────────────────
ohe_names = pipe['prep'].named_transformers_['cat'].get_feature_names_out(CAT_FEATURES)
all_names = np.array(NUM_FEATURES + list(ohe_names))
coef_df   = (pd.DataFrame({'feature': all_names, 'coefficient': pipe['model'].coef_})
               .assign(abs_coef=lambda d: d['coefficient'].abs())
               .sort_values('abs_coef', ascending=False)
               .head(20).reset_index(drop=True))

# ── Per-stratum performance ───────────────────────────────────────────────────
meta_test = meta_test.reset_index(drop=True)
resid_df  = pd.DataFrame({
    'actual':        y_test.values,
    'predicted':     y_pred_test,
    'residual':      y_test.values - y_pred_test,
    'weather_group': meta_test['weather_group'].values,
    'cab_type':      meta_test['cab_type'].values,
    'time_period':   meta_test['time_period'].values,
})

def perf_by(col, order=None):
    rows = []
    for grp, sub in resid_df.groupby(col):
        m = metrics(sub['actual'], sub['predicted'])
        rows.append({col: grp, **m})
    out = pd.DataFrame(rows).sort_values('MAE')
    if order:
        out = out.set_index(col).reindex(order).reset_index()
    return out

perf_weather = perf_by('weather_group')
perf_cab     = perf_by('cab_type')
perf_time    = perf_by('time_period', TIME_ORDER)

# ── Main results figure ───────────────────────────────────────────────────────
sample_idx = np.random.default_rng(42).integers(0, len(y_test), 4000)
residuals  = y_test.values - y_pred_test

fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=(
        'Top 20 feature coefficients',
        'Actual vs Predicted (test set)',
        'Residuals distribution',
        'Residuals vs Fitted',
        'MAE by weather condition',
        'MAE by cab type & time of day',
    ),
    vertical_spacing=0.12,
    horizontal_spacing=0.12,
    row_heights=[0.34, 0.34, 0.32],
)

# Coefficients
colors = ['#E45756' if c < 0 else '#4C78A8' for c in coef_df['coefficient']]
fig.add_trace(go.Bar(
    x=coef_df['coefficient'], y=coef_df['feature'],
    orientation='h', marker_color=colors,
    showlegend=False,
    hovertemplate='%{y}: %{x:.3f}<extra></extra>',
), row=1, col=1)
fig.add_vline(x=0, line_width=1, line_color='grey', row=1, col=1)

# Actual vs Predicted
fig.add_trace(go.Scatter(
    x=y_test.values[sample_idx], y=y_pred_test[sample_idx],
    mode='markers',
    marker=dict(size=3, color='#4C78A8', opacity=0.3),
    showlegend=False,
    hovertemplate='Actual: $%{x:.2f}<br>Predicted: $%{y:.2f}<extra></extra>',
), row=1, col=2)
lim = [0, float(y_test.max())]
fig.add_trace(go.Scatter(
    x=lim, y=lim, mode='lines',
    line=dict(color='#E45756', dash='dash', width=1.5),
    showlegend=False,
), row=1, col=2)

# Residuals histogram
fig.add_trace(go.Histogram(
    x=residuals, nbinsx=80,
    marker_color='#4C78A8', opacity=0.75,
    showlegend=False,
), row=2, col=1)
fig.add_vline(x=0, line_width=1.5, line_dash='dash', line_color='#E45756', row=2, col=1)

# Residuals vs Fitted
fig.add_trace(go.Scatter(
    x=y_pred_test[sample_idx], y=residuals[sample_idx],
    mode='markers',
    marker=dict(size=3, color='#4C78A8', opacity=0.3),
    showlegend=False,
), row=2, col=2)
fig.add_hline(y=0, line_width=1.5, line_dash='dash', line_color='#E45756', row=2, col=2)

# MAE by weather
fig.add_trace(go.Bar(
    x=perf_weather['MAE'], y=perf_weather['weather_group'],
    orientation='h',
    marker_color='#4C78A8', opacity=0.8,
    text=[f'${v:.2f}' for v in perf_weather['MAE']],
    textposition='outside',
    showlegend=False,
    hovertemplate='%{y}<br>MAE: $%{x:.2f}<extra></extra>',
), row=3, col=1)

# MAE by cab type
for cab, color in [('Uber', '#1a1a2e'), ('Lyft', '#E91E8C')]:
    sub = perf_time.copy()
    cab_row = perf_cab[perf_cab['cab_type'] == cab].iloc[0]
    fig.add_trace(go.Bar(
        x=[cab_row['MAE']], y=[cab],
        orientation='h',
        name=cab,
        marker_color=color,
        text=[f'${cab_row["MAE"]:.2f}'],
        textposition='outside',
        showlegend=False,
    ), row=3, col=2)

# MAE by time of day (overlay as scatter on same axis)
for _, row_t in perf_time.iterrows():
    fig.add_trace(go.Scatter(
        x=[row_t['MAE']], y=[row_t['time_period']],
        mode='markers+text',
        marker=dict(size=10, color='#F58518'),
        text=[f'${row_t["MAE"]:.2f}'],
        textposition='middle right',
        showlegend=False,
        hovertemplate=f'{row_t["time_period"]}<br>MAE: ${row_t["MAE"]:.2f}<extra></extra>',
    ), row=3, col=2)

fig.update_layout(
    title=dict(
        text=(
            f'<b>Linear Regression — Stratified Split</b><br>'
            f'<sup>Stratified by: weather condition × cab type × time of day &nbsp;|&nbsp;'
            f'Test R²: {m_test["R2"]:.4f} &nbsp;|&nbsp;'
            f'MAE: ${m_test["MAE"]:.2f} &nbsp;|&nbsp;'
            f'RMSE: ${m_test["RMSE"]:.2f}</sup>'
        ),
        font=dict(size=8),
    ),
    height=1800,
    margin=dict(t=10, b=60, l=180, r=80),
    plot_bgcolor='white',
    paper_bgcolor='white',
)

fig.update_xaxes(showgrid=True, gridcolor='#eee')
fig.update_yaxes(showgrid=True, gridcolor='#eee')
fig.update_xaxes(title_text='Coefficient',       row=1, col=1)
fig.update_xaxes(title_text='Actual ($)',         row=1, col=2)
fig.update_yaxes(title_text='Predicted ($)',      row=1, col=2)
fig.update_xaxes(title_text='Residual ($)',        row=2, col=1)
fig.update_xaxes(title_text='Fitted ($)',          row=2, col=2)
fig.update_yaxes(title_text='Residual ($)',        row=2, col=2)
fig.update_xaxes(title_text='MAE ($)',             row=3, col=1)
fig.update_xaxes(title_text='MAE ($)',             row=3, col=2)

fig.show()

# ── Per-stratum summary ───────────────────────────────────────────────────────
print("\nMAE by weather condition:")
print(perf_weather[['weather_group','MAE','RMSE','R2']].to_string(index=False))
print("\nMAE by cab type:")
print(perf_cab[['cab_type','MAE','RMSE','R2']].to_string(index=False))
print("\nMAE by time of day:")
print(perf_time[['time_period','MAE','RMSE','R2']].to_string(index=False))

In [ ]:
from sklearn.linear_model import RidgeCV
if 'day_of_week' not in df.columns:
    df['day_of_week'] = pd.to_datetime(df['datetime']).dt.day_name()
if 'is_weekend' not in df.columns:
    df['is_weekend'] = df['day_of_week'].isin(['Saturday', 'Sunday']).astype(int)

def time_bucket(h):
    if   6 <= h < 10: return 'Morning Commute'
    elif 10 <= h < 16: return 'Daytime'
    elif 16 <= h < 20: return 'Evening Commute'
    elif 20 <= h < 24: return 'Nightlife'
    else:              return 'Late Night'

if 'time_period' not in df.columns:
    df['time_period'] = df['hour'].apply(time_bucket)

# ── Stratification key (same as before) ──────────────────────────────────────
MIN_WEATHER_COUNT = 1000
weather_counts    = df['short_summary'].value_counts()
common_weather    = weather_counts[weather_counts >= MIN_WEATHER_COUNT].index
df['weather_group'] = df['short_summary'].where(df['short_summary'].isin(common_weather), 'Other')

df['strat_key'] = (
    df['weather_group'] + '__' +
    df['cab_type']      + '__' +
    df['time_period']
)

key_counts = df['strat_key'].value_counts()
valid_keys = key_counts[key_counts >= 20].index
df_model   = df[df['strat_key'].isin(valid_keys)].copy()

# ── Features ──────────────────────────────────────────────────────────────────
NUM_FEATURES = ['distance', 'surge_multiplier', 'hour', 'is_weekend',
                'temperature', 'precipProbability', 'windSpeed']
CAT_FEATURES = ['name', 'cab_type', 'source', 'destination']
TARGET       = 'price'
TIME_ORDER   = ['Morning Commute', 'Daytime', 'Evening Commute', 'Nightlife', 'Late Night']

df_model = df_model[NUM_FEATURES + CAT_FEATURES + [TARGET,
           'strat_key', 'weather_group', 'time_period']].dropna()

X = df_model[NUM_FEATURES + CAT_FEATURES]
y = df_model[TARGET]

# ── Stratified split ──────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, df_model.index,
    test_size=0.2,
    random_state=42,
    stratify=df_model['strat_key'],
)
print(f"Train: {len(X_train):,}  |  Test: {len(X_test):,}")

meta_test = df_model.loc[idx_test, ['weather_group', 'cab_type', 'time_period']].copy()

# ── Stratification check ──────────────────────────────────────────────────────
def split_dist(col):
    tr = df_model.loc[idx_train, col].value_counts(normalize=True).rename('Train')
    te = df_model.loc[idx_test,  col].value_counts(normalize=True).rename('Test')
    return pd.concat([tr, te], axis=1).reset_index().rename(columns={'index': col})

dist_weather = split_dist('weather_group')
dist_cab     = split_dist('cab_type')
dist_time    = split_dist('time_period')

fig_strat = make_subplots(
    rows=1, cols=3,
    subplot_titles=('Weather condition', 'Cab type', 'Time of day'),
    horizontal_spacing=0.1,
)
for col, dist, order, ci in [
    ('weather_group', dist_weather, None,       1),
    ('cab_type',      dist_cab,     None,       2),
    ('time_period',   dist_time,    TIME_ORDER, 3),
]:
    if order:
        dist = dist.set_index(col).reindex(order).reset_index()
    for split, color in [('Train', '#4C78A8'), ('Test', '#E45756')]:
        fig_strat.add_trace(go.Bar(
            x=dist[col], y=dist[split],
            name=split, marker_color=color, opacity=0.75,
            showlegend=(ci == 1),
            hovertemplate=f'{split}: %{{y:.2%}}<extra></extra>',
        ), row=1, col=ci)

fig_strat.update_layout(
    title='<b>Stratification check — Train vs Test distributions</b>',
    barmode='group', height=380,
    margin=dict(t=80, b=60),
    plot_bgcolor='white', paper_bgcolor='white',
    legend=dict(title='Split'),
)
fig_strat.update_yaxes(tickformat='.0%', showgrid=True, gridcolor='#eee')
fig_strat.update_xaxes(tickangle=20)
fig_strat.show()

# ── Ridge with cross-validated alpha ─────────────────────────────────────────
# Search over a wide log-spaced grid; RidgeCV does efficient LOO/GCV internally
ALPHAS = np.logspace(-2, 5, 80)

preprocessor = ColumnTransformer([
    ('num', StandardScaler(),                                             NUM_FEATURES),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False),  CAT_FEATURES),
])

pipe = Pipeline([
    ('prep',  preprocessor),
    ('model', RidgeCV(alphas=ALPHAS, cv=5, scoring='neg_mean_absolute_error')),
])

pipe.fit(X_train, y_train)

best_alpha = pipe['model'].alpha_
print(f"\nBest alpha (CV-selected): {best_alpha:.4f}")

# ── Predictions & metrics ─────────────────────────────────────────────────────
y_pred_train = pipe.predict(X_train)
y_pred_test  = pipe.predict(X_test)

def metrics(y_true, y_pred):
    return dict(
        MAE  = mean_absolute_error(y_true, y_pred),
        RMSE = mean_squared_error(y_true,  y_pred) ** 0.5,
        R2   = r2_score(y_true,            y_pred),
    )

m_train = metrics(y_train, y_pred_train)
m_test  = metrics(y_test,  y_pred_test)

print(f"\n{'Metric':<6} {'Train':>12} {'Test':>12}")
print("-" * 32)
for m in ['MAE', 'RMSE', 'R2']:
    fmt = '.4f' if m == 'R2' else '.2f'
    print(f"{m:<6} {m_train[m]:>12{fmt}} {m_test[m]:>12{fmt}}")

# ── Alpha sensitivity curve ───────────────────────────────────────────────────
# Refit with each alpha to show how MAE changes (on a subsample for speed)
from sklearn.linear_model import Ridge

sample_n  = min(30000, len(X_train))
rng       = np.random.default_rng(42)
s_idx     = rng.integers(0, len(X_train), sample_n)
X_tr_s    = X_train.iloc[s_idx]
y_tr_s    = y_train.iloc[s_idx]

# Preprocess once
X_tr_pre  = preprocessor.transform(X_train)
X_te_pre  = preprocessor.transform(X_test)
X_tr_s_pre = preprocessor.transform(X_tr_s)

alpha_maes_train, alpha_maes_test = [], []
ALPHA_PLOT = np.logspace(-2, 5, 40)
for a in ALPHA_PLOT:
    r = Ridge(alpha=a).fit(X_tr_pre, y_train)
    alpha_maes_train.append(mean_absolute_error(y_train, r.predict(X_tr_pre)))
    alpha_maes_test.append( mean_absolute_error(y_test,  r.predict(X_te_pre)))

# ── Coefficients ──────────────────────────────────────────────────────────────
ohe_names = pipe['prep'].named_transformers_['cat'].get_feature_names_out(CAT_FEATURES)
all_names = np.array(NUM_FEATURES + list(ohe_names))
coef_df   = (pd.DataFrame({'feature': all_names, 'coefficient': pipe['model'].coef_})
               .assign(abs_coef=lambda d: d['coefficient'].abs())
               .sort_values('abs_coef', ascending=False)
               .head(20).reset_index(drop=True))

# ── Per-stratum performance ───────────────────────────────────────────────────
meta_test = meta_test.reset_index(drop=True)
resid_df  = pd.DataFrame({
    'actual':        y_test.values,
    'predicted':     y_pred_test,
    'residual':      y_test.values - y_pred_test,
    'weather_group': meta_test['weather_group'].values,
    'cab_type':      meta_test['cab_type'].values,
    'time_period':   meta_test['time_period'].values,
})

def perf_by(col, order=None):
    rows = []
    for grp, sub in resid_df.groupby(col):
        m = metrics(sub['actual'], sub['predicted'])
        rows.append({col: grp, **m})
    out = pd.DataFrame(rows).sort_values('MAE')
    if order:
        out = out.set_index(col).reindex(order).reset_index()
    return out

perf_weather = perf_by('weather_group')
perf_cab     = perf_by('cab_type')
perf_time    = perf_by('time_period', TIME_ORDER)

# ── Main figure ───────────────────────────────────────────────────────────────
sample_idx = rng.integers(0, len(y_test), 4000)
residuals  = y_test.values - y_pred_test

fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=(
        'Top 20 feature coefficients (Ridge-shrunk)',
        'Alpha sensitivity — MAE vs regularisation strength',
        'Residuals distribution',
        'Actual vs Predicted (test set)',
        'MAE by weather condition',
        'MAE by cab type & time of day',
    ),
    vertical_spacing=0.12,
    horizontal_spacing=0.12,
    row_heights=[0.34, 0.34, 0.32],
)

# Coefficients
colors = ['#E45756' if c < 0 else '#4C78A8' for c in coef_df['coefficient']]
fig.add_trace(go.Bar(
    x=coef_df['coefficient'], y=coef_df['feature'],
    orientation='h', marker_color=colors,
    showlegend=False,
    hovertemplate='%{y}: %{x:.4f}<extra></extra>',
), row=1, col=1)
fig.add_vline(x=0, line_width=1, line_color='grey', row=1, col=1)

# Alpha sensitivity
fig.add_trace(go.Scatter(
    x=ALPHA_PLOT, y=alpha_maes_train,
    mode='lines', name='Train MAE',
    line=dict(color='#4C78A8', width=2),
), row=1, col=2)
fig.add_trace(go.Scatter(
    x=ALPHA_PLOT, y=alpha_maes_test,
    mode='lines', name='Test MAE',
    line=dict(color='#E45756', width=2),
), row=1, col=2)
fig.add_vline(
    x=best_alpha, line_dash='dash', line_color='#54A24B', line_width=1.5,
    annotation_text=f'α={best_alpha:.2f}', annotation_position='top right',
    row=1, col=2,
)

# Residuals histogram
fig.add_trace(go.Histogram(
    x=residuals, nbinsx=80,
    marker_color='#4C78A8', opacity=0.75,
    showlegend=False,
), row=2, col=1)
fig.add_vline(x=0, line_width=1.5, line_dash='dash', line_color='#E45756', row=2, col=1)

# Actual vs Predicted
fig.add_trace(go.Scatter(
    x=y_test.values[sample_idx], y=y_pred_test[sample_idx],
    mode='markers',
    marker=dict(size=3, color='#4C78A8', opacity=0.3),
    showlegend=False,
    hovertemplate='Actual: $%{x:.2f}<br>Predicted: $%{y:.2f}<extra></extra>',
), row=2, col=2)
lim = [0, float(y_test.max())]
fig.add_trace(go.Scatter(
    x=lim, y=lim, mode='lines',
    line=dict(color='#E45756', dash='dash', width=1.5),
    showlegend=False,
), row=2, col=2)

# MAE by weather
fig.add_trace(go.Bar(
    x=perf_weather['MAE'], y=perf_weather['weather_group'],
    orientation='h', marker_color='#4C78A8', opacity=0.8,
    text=[f'${v:.2f}' for v in perf_weather['MAE']],
    textposition='outside', showlegend=False,
    hovertemplate='%{y}<br>MAE: $%{x:.2f}<extra></extra>',
), row=3, col=1)

# MAE by cab type
for cab, color in [('Uber', '#1a1a2e'), ('Lyft', '#E91E8C')]:
    cab_row = perf_cab[perf_cab['cab_type'] == cab].iloc[0]
    fig.add_trace(go.Bar(
        x=[cab_row['MAE']], y=[cab],
        orientation='h', name=cab, marker_color=color,
        text=[f'${cab_row["MAE"]:.2f}'], textposition='outside',
        showlegend=False,
    ), row=3, col=2)

# MAE by time of day
for _, row_t in perf_time.iterrows():
    fig.add_trace(go.Scatter(
        x=[row_t['MAE']], y=[row_t['time_period']],
        mode='markers+text',
        marker=dict(size=10, color='#F58518'),
        text=[f'${row_t["MAE"]:.2f}'],
        textposition='middle right',
        showlegend=False,
    ), row=3, col=2)

fig.update_layout(
    title=dict(
        text=(
            f'<b>Ridge Regression — Stratified Split</b><br>'
            f'<sup>Stratified by: weather × cab type × time of day &nbsp;|&nbsp;'
            f'Best α: {best_alpha:.4f} (CV-selected) &nbsp;|&nbsp;'
            f'Test R²: {m_test["R2"]:.4f} &nbsp;|&nbsp;'
            f'MAE: ${m_test["MAE"]:.2f} &nbsp;|&nbsp;'
            f'RMSE: ${m_test["RMSE"]:.2f}</sup>'
        ),
        font=dict(size=8),
    ),
    height=1800,
    margin=dict(t=10, b=6, l=18, r=8),
    plot_bgcolor='white', paper_bgcolor='white',
    legend=dict(x=0.55, y=0.68, title=''),
)
fig.update_xaxes(showgrid=True, gridcolor='#eee')
fig.update_yaxes(showgrid=True, gridcolor='#eee')
fig.update_xaxes(title_text='Coefficient',   row=1, col=1)
fig.update_xaxes(title_text='Alpha (log scale)', type='log', row=1, col=2)
fig.update_yaxes(title_text='MAE ($)',           row=1, col=2)
fig.update_xaxes(title_text='Residual ($)',      row=2, col=1)
fig.update_xaxes(title_text='Actual ($)',        row=2, col=2)
fig.update_yaxes(title_text='Predicted ($)',     row=2, col=2)
fig.update_xaxes(title_text='MAE ($)',           row=3, col=1)
fig.update_xaxes(title_text='MAE ($)',           row=3, col=2)

fig.show()

print("\nMAE by weather condition:")
print(perf_weather[['weather_group','MAE','RMSE','R2']].to_string(index=False))
print("\nMAE by cab type:")
print(perf_cab[['cab_type','MAE','RMSE','R2']].to_string(index=False))
print("\nMAE by time of day:")
print(perf_time[['time_period','MAE','RMSE','R2']].to_string(index=False))

In [ ]:
!pip install lightgbm

In [ ]:
import lightgbm as lgb
import xgboost as xgb
# ── Derived columns ───────────────────────────────────────────────────────────
if 'day_of_week' not in df.columns:
    df['day_of_week'] = pd.to_datetime(df['datetime']).dt.day_name()
if 'is_weekend' not in df.columns:
    df['is_weekend'] = df['day_of_week'].isin(['Saturday', 'Sunday']).astype(int)

def time_bucket(h):
    if   6 <= h < 10: return 'Morning Commute'
    elif 10 <= h < 16: return 'Daytime'
    elif 16 <= h < 20: return 'Evening Commute'
    elif 20 <= h < 24: return 'Nightlife'
    else:              return 'Late Night'

if 'time_period' not in df.columns:
    df['time_period'] = df['hour'].apply(time_bucket)

# ── Stratification key ────────────────────────────────────────────────────────
MIN_WEATHER_COUNT = 1000
weather_counts    = df['short_summary'].value_counts()
common_weather    = weather_counts[weather_counts >= MIN_WEATHER_COUNT].index
df['weather_group'] = df['short_summary'].where(df['short_summary'].isin(common_weather), 'Other')

df['strat_key'] = (
    df['weather_group'] + '__' +
    df['cab_type']      + '__' +
    df['time_period']
)
key_counts = df['strat_key'].value_counts()
df_model   = df[df['strat_key'].isin(key_counts[key_counts >= 20].index)].copy()

NUM_FEATURES = ['distance', 'surge_multiplier', 'hour', 'is_weekend',
                'temperature', 'precipProbability', 'windSpeed']
CAT_FEATURES = ['name', 'cab_type', 'source', 'destination']
TARGET       = 'price'
TIME_ORDER   = ['Morning Commute', 'Daytime', 'Evening Commute', 'Nightlife', 'Late Night']

df_model = df_model[NUM_FEATURES + CAT_FEATURES + [TARGET,
           'strat_key', 'weather_group', 'time_period']].dropna()

X = df_model[NUM_FEATURES + CAT_FEATURES]
y = df_model[TARGET]

# ── Stratified train / test split ─────────────────────────────────────────────
X_train_full, X_test, y_train_full, y_test, idx_train_full, idx_test = train_test_split(
    X, y, df_model.index,
    test_size=0.2, random_state=42,
    stratify=df_model['strat_key'],
)

# Carve out a validation set from train for early stopping (10% of full data)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.125, random_state=42,     # 0.125 × 0.8 = 0.10 of full data
    stratify=df_model.loc[idx_train_full, 'strat_key'],
)
print(f"Train: {len(X_train):,}  Val: {len(X_val):,}  Test: {len(X_test):,}")

meta_test = df_model.loc[idx_test, ['weather_group', 'cab_type', 'time_period']].reset_index(drop=True)

# ── Preprocessing (fit once, reuse for both models) ───────────────────────────
preprocessor = ColumnTransformer([
    ('num', StandardScaler(),                                             NUM_FEATURES),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False),  CAT_FEATURES),
])

X_train_pre = preprocessor.fit_transform(X_train)
X_val_pre   = preprocessor.transform(X_val)
X_test_pre  = preprocessor.transform(X_test)

ohe_names     = preprocessor.named_transformers_['cat'].get_feature_names_out(CAT_FEATURES)
feature_names = NUM_FEATURES + list(ohe_names)

# ─────────────────────────────────────────────────────────────────────────────
# LightGBM
# ─────────────────────────────────────────────────────────────────────────────
lgb_model = lgb.LGBMRegressor(
    n_estimators     = 2000,
    learning_rate    = 0.05,
    num_leaves       = 63,
    max_depth        = 7,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    min_child_samples= 50,
    reg_alpha        = 0.1,
    reg_lambda       = 1.0,
    random_state     = 42,
    n_jobs           = -1,
    verbose          = -1,
)

lgb_model.fit(
    X_train_pre, y_train,
    eval_set   = [(X_train_pre, y_train), (X_val_pre, y_val)],
    eval_names = ['train', 'val'],
    eval_metric= 'mae',
    callbacks  = [
        lgb.early_stopping(stopping_rounds=50, verbose=False),
        lgb.log_evaluation(period=100),
    ],
)

lgb_history = lgb_model.evals_result_
lgb_best_n  = lgb_model.best_iteration_
print(f"\nLightGBM best iteration: {lgb_best_n}")

lgb_pred_test  = lgb_model.predict(X_test_pre)
lgb_pred_train = lgb_model.predict(X_train_pre)

# ─────────────────────────────────────────────────────────────────────────────
# XGBoost
# ─────────────────────────────────────────────────────────────────────────────
xgb_model = xgb.XGBRegressor(
    n_estimators        = 2000,
    learning_rate       = 0.05,
    max_depth           = 7,
    subsample           = 0.8,
    colsample_bytree    = 0.8,
    min_child_weight    = 50,
    reg_alpha           = 0.1,
    reg_lambda          = 1.0,
    early_stopping_rounds= 50,
    eval_metric         = 'mae',
    random_state        = 42,
    n_jobs              = -1,
    verbosity           = 0,
)

xgb_model.fit(
    X_train_pre, y_train,
    eval_set   = [(X_train_pre, y_train), (X_val_pre, y_val)],
    verbose    = 100,
)

xgb_history = xgb_model.evals_result()
xgb_best_n  = xgb_model.best_iteration
print(f"XGBoost  best iteration: {xgb_best_n}")

xgb_pred_test  = xgb_model.predict(X_test_pre)
xgb_pred_train = xgb_model.predict(X_train_pre)

# ─────────────────────────────────────────────────────────────────────────────
# Metrics
# ─────────────────────────────────────────────────────────────────────────────
def metrics(y_true, y_pred):
    return dict(
        MAE  = mean_absolute_error(y_true, y_pred),
        RMSE = mean_squared_error(y_true,  y_pred) ** 0.5,
        R2   = r2_score(y_true,            y_pred),
    )

results = {
    'LightGBM': {'train': metrics(y_train, lgb_pred_train), 'test': metrics(y_test, lgb_pred_test)},
    'XGBoost':  {'train': metrics(y_train, xgb_pred_train), 'test': metrics(y_test, xgb_pred_test)},
}

print(f"\n{'Model':<12} {'Train MAE':>10} {'Test MAE':>10} {'Train R²':>10} {'Test R²':>10} {'RMSE':>10}")
print("-" * 55)
for model, res in results.items():
    print(f"{model:<12} {res['train']['MAE']:>10.2f} {res['test']['MAE']:>10.2f} "
          f"{res['train']['R2']:>10.4f} {res['test']['R2']:>10.4f} {res['test']['RMSE']:>10.2f}")

# ── Per-stratum performance ───────────────────────────────────────────────────
def perf_by(col, pred_lgb, pred_xgb, order=None):
    rows = []
    for grp in meta_test[col].unique():
        mask = meta_test[col] == grp
        rows.append({
            col:         grp,
            'LGB_MAE':   mean_absolute_error(y_test.values[mask], pred_lgb[mask]),
            'XGB_MAE':   mean_absolute_error(y_test.values[mask], pred_xgb[mask]),
        })
    out = pd.DataFrame(rows).sort_values('LGB_MAE')
    if order:
        out = out.set_index(col).reindex(order).reset_index()
    return out

pw = perf_by('weather_group', lgb_pred_test, xgb_pred_test)
pc = perf_by('cab_type',      lgb_pred_test, xgb_pred_test)
pt = perf_by('time_period',   lgb_pred_test, xgb_pred_test, TIME_ORDER)

# ── Feature importance (top 20) ───────────────────────────────────────────────
def top_importance(model, n=20):
    return (pd.DataFrame({'feature': feature_names, 'importance': model.feature_importances_})
              .sort_values('importance', ascending=False).head(n).reset_index(drop=True))

lgb_imp = top_importance(lgb_model)
xgb_imp = top_importance(xgb_model)

# ─────────────────────────────────────────────────────────────────────────────
# Figure 1 — Learning curves
# ─────────────────────────────────────────────────────────────────────────────
fig_lc = make_subplots(
    rows=1, cols=2,
    subplot_titles=('LightGBM — MAE vs rounds', 'XGBoost — MAE vs rounds'),
    horizontal_spacing=0.1,
)

lgb_metric = list(lgb_history['train'].keys())[0]

for col_i, (history, best_n, key_train, key_val, metric_key, label) in enumerate([
    (lgb_history, lgb_best_n, 'train',        'val',          lgb_metric, 'LightGBM'),
    (xgb_history, xgb_best_n, 'validation_0', 'validation_1', 'mae',      'XGBoost'),
], start=1):
    tr_mae = history[key_train][metric_key]
    va_mae = history[key_val][metric_key]
    rounds = list(range(1, len(tr_mae) + 1))

    fig_lc.add_trace(go.Scatter(
        x=rounds, y=tr_mae, mode='lines', name='Train',
        line=dict(color='#4C78A8', width=1.5),
        showlegend=(col_i == 1),
    ), row=1, col=col_i)
    fig_lc.add_trace(go.Scatter(
        x=rounds, y=va_mae, mode='lines', name='Validation',
        line=dict(color='#E45756', width=1.5),
        showlegend=(col_i == 1),
    ), row=1, col=col_i)
    fig_lc.add_vline(
        x=best_n, line_dash='dash', line_color='#54A24B', line_width=1.5,
        annotation_text=f'Best: {best_n}',
        annotation_position='top right',
        row=1, col=col_i,
    )

fig_lc.update_layout(
    title='<b>Learning Curves — Early Stopping</b><br>'
          '<sup>Green dashed = best iteration selected by early stopping</sup>',
    height=380, margin=dict(t=90, b=60),
    plot_bgcolor='white', paper_bgcolor='white',
    legend=dict(title='Split'),
)
fig_lc.update_xaxes(title_text='Boosting rounds', showgrid=True, gridcolor='#eee')
fig_lc.update_yaxes(title_text='MAE ($)',          showgrid=True, gridcolor='#eee')
fig_lc.show()

# ─────────────────────────────────────────────────────────────────────────────
# Figure 2 — Head-to-head comparison
# ─────────────────────────────────────────────────────────────────────────────
rng        = np.random.default_rng(42)
sample_idx = rng.integers(0, len(y_test), 3000)
MODEL_COLORS = {'LightGBM': '#4C78A8', 'XGBoost': '#E45756'}

fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=(
        'LightGBM — top 20 feature importances',
        'XGBoost  — top 20 feature importances',
        'Actual vs Predicted',
        'Residual distributions',
        'MAE by weather condition',
        'MAE by time of day',
    ),
    vertical_spacing=0.12,
    horizontal_spacing=0.12,
    row_heights=[0.35, 0.33, 0.32],
)

# Feature importance — LGB
fig.add_trace(go.Bar(
    x=lgb_imp['importance'], y=lgb_imp['feature'],
    orientation='h', marker_color='#4C78A8',
    showlegend=False,
    hovertemplate='%{y}: %{x:.0f}<extra>LightGBM</extra>',
), row=1, col=1)

# Feature importance — XGB
fig.add_trace(go.Bar(
    x=xgb_imp['importance'], y=xgb_imp['feature'],
    orientation='h', marker_color='#E45756',
    showlegend=False,
    hovertemplate='%{y}: %{x:.0f}<extra>XGBoost</extra>',
), row=1, col=2)

# Actual vs Predicted — both models
for name, pred, color in [
    ('LightGBM', lgb_pred_test, '#4C78A8'),
    ('XGBoost',  xgb_pred_test, '#E45756'),
]:
    fig.add_trace(go.Scatter(
        x=y_test.values[sample_idx], y=pred[sample_idx],
        mode='markers', name=name,
        marker=dict(size=3, color=color, opacity=0.3),
        hovertemplate=f'<b>{name}</b><br>Actual: $%{{x:.2f}}<br>Pred: $%{{y:.2f}}<extra></extra>',
    ), row=2, col=1)

lim = [0, float(y_test.max())]
fig.add_trace(go.Scatter(
    x=lim, y=lim, mode='lines',
    line=dict(color='grey', dash='dash', width=1.2),
    showlegend=False,
), row=2, col=1)

# Residuals — both models
for name, pred, color in [
    ('LightGBM', lgb_pred_test, '#4C78A8'),
    ('XGBoost',  xgb_pred_test, '#E45756'),
]:
    fig.add_trace(go.Histogram(
        x=y_test.values - pred,
        nbinsx=100, name=name,
        marker_color=color, opacity=0.55,
        histnorm='probability density',
        showlegend=False,
        hovertemplate=f'{name} residual: %{{x:.2f}}<extra></extra>',
    ), row=2, col=2)
fig.add_vline(x=0, line_width=1.5, line_dash='dash', line_color='grey', row=2, col=2)

# MAE by weather — grouped bars
for model_name, col_name, color in [
    ('LightGBM', 'LGB_MAE', '#4C78A8'),
    ('XGBoost',  'XGB_MAE', '#E45756'),
]:
    fig.add_trace(go.Bar(
        x=pw[col_name], y=pw['weather_group'],
        orientation='h', name=model_name,
        marker_color=color, opacity=0.8,
        showlegend=True,
        hovertemplate=f'%{{y}}<br>{model_name} MAE: $%{{x:.2f}}<extra></extra>',
    ), row=3, col=1)

# MAE by time of day — grouped bars
for model_name, col_name, color in [
    ('LightGBM', 'LGB_MAE', '#4C78A8'),
    ('XGBoost',  'XGB_MAE', '#E45756'),
]:
    fig.add_trace(go.Bar(
        x=pt[col_name], y=pt['time_period'],
        orientation='h', name=model_name,
        marker_color=color, opacity=0.8,
        showlegend=False,
        hovertemplate=f'%{{y}}<br>{model_name} MAE: $%{{x:.2f}}<extra></extra>',
    ), row=3, col=2)

fig.update_layout(
    title=dict(
        text=(
            '<b>LightGBM vs XGBoost — Stratified Split Comparison</b><br>'
            f'<sup>'
            f'LightGBM → Test MAE: ${results["LightGBM"]["test"]["MAE"]:.2f} &nbsp;|&nbsp;'
            f'R²: {results["LightGBM"]["test"]["R2"]:.4f} &nbsp;|&nbsp;'
            f'Best iter: {lgb_best_n}'
            f'&nbsp;&nbsp;&nbsp;&nbsp;'
            f'XGBoost  → Test MAE: ${results["XGBoost"]["test"]["MAE"]:.2f} &nbsp;|&nbsp;'
            f'R²: {results["XGBoost"]["test"]["R2"]:.4f} &nbsp;|&nbsp;'
            f'Best iter: {xgb_best_n}'
            f'</sup>'
        ),
        font=dict(size=14),
    ),
    height=1100,
    barmode='group',
    margin=dict(t=110, b=60, l=200, r=60),
    plot_bgcolor='white', paper_bgcolor='white',
    legend=dict(title='Model', x=1.01, y=0.55),
)
fig.update_xaxes(showgrid=True, gridcolor='#eee')
fig.update_yaxes(showgrid=True, gridcolor='#eee')
fig.update_xaxes(title_text='Importance',            row=1, col=1)
fig.update_xaxes(title_text='Importance',            row=1, col=2)
fig.update_xaxes(title_text='Actual price ($)',       row=2, col=1)
fig.update_yaxes(title_text='Predicted price ($)',    row=2, col=1)
fig.update_xaxes(title_text='Residual ($)',           row=2, col=2)
fig.update_yaxes(title_text='Density',               row=2, col=2)
fig.update_xaxes(title_text='MAE ($)',               row=3, col=1)
fig.update_xaxes(title_text='MAE ($)',               row=3, col=2)
fig.show()

# ─────────────────────────────────────────────────────────────────────────────
# Figure 3 — Overall metrics bar chart (all 4 models if prior results exist)
# ─────────────────────────────────────────────────────────────────────────────
all_results = {
    'LightGBM': results['LightGBM']['test'],
    'XGBoost':  results['XGBoost']['test'],
}
# Uncomment and fill if you stored metrics from earlier models:
# all_results['Linear'] = {'MAE': ..., 'RMSE': ..., 'R2': ...}
# all_results['Ridge']  = {'MAE': ..., 'RMSE': ..., 'R2': ...}

fig_cmp = make_subplots(
    rows=1, cols=3,
    subplot_titles=('MAE (lower = better)', 'RMSE (lower = better)', 'R² (higher = better)'),
    horizontal_spacing=0.1,
)

PALETTE = ['#4C78A8', '#E45756', '#F58518', '#54A24B']
model_names = list(all_results.keys())

for ci, metric in enumerate(['MAE', 'RMSE', 'R2'], start=1):
    vals = [all_results[m][metric] for m in model_names]
    best = min(vals) if metric != 'R2' else max(vals)
    colors = ['#54A24B' if v == best else PALETTE[i] for i, v in enumerate(vals)]
    fig_cmp.add_trace(go.Bar(
        x=model_names, y=vals,
        marker_color=colors,
        text=[f'{v:.4f}' if metric == 'R2' else f'${v:.2f}' for v in vals],
        textposition='outside',
        showlegend=False,
        hovertemplate='%{x}: %{y:.4f}<extra></extra>',
    ), row=1, col=ci)

fig_cmp.update_layout(
    title='<b>Model Comparison — Test Set Metrics</b><br>'
          '<sup>Green bar = best performer on each metric</sup>',
    height=420,
    margin=dict(t=90, b=60),
    plot_bgcolor='white', paper_bgcolor='white',
)
fig_cmp.update_xaxes(showgrid=False)
fig_cmp.update_yaxes(showgrid=True, gridcolor='#eee')
fig_cmp.show()

# ── Stratum breakdown ─────────────────────────────────────────────────────────
print("\nMAE by weather condition:")
print(pw.to_string(index=False))
print("\nMAE by cab type:")
print(pc.to_string(index=False))
print("\nMAE by time of day:")
print(pt.to_string(index=False))

In [ ]:
import shap
# ── Sample for SHAP (2k points is fast and representative) ───────────────────
SHAP_N   = 2000
rng      = np.random.default_rng(42)
shap_idx = rng.integers(0, X_test_pre.shape[0], SHAP_N)
X_shap   = X_test_pre[shap_idx]
y_shap   = y_test.values[shap_idx]

print("Computing LightGBM SHAP values...")
lgb_exp  = shap.TreeExplainer(lgb_model)
lgb_sv   = lgb_exp.shap_values(X_shap)
lgb_base = float(lgb_exp.expected_value)

print("Computing XGBoost SHAP values...")
xgb_exp  = shap.TreeExplainer(xgb_model)
xgb_sv   = xgb_exp.shap_values(X_shap)
xgb_base = float(xgb_exp.expected_value)

lgb_pred_shap = lgb_model.predict(X_shap)
xgb_pred_shap = xgb_model.predict(X_shap)

print(f"\nBase value  LGB: ${lgb_base:.2f}  |  XGB: ${xgb_base:.2f}")

N_TOP = 15

def top_idx(sv, n=N_TOP):
    return np.argsort(np.abs(sv).mean(axis=0))[::-1][:n]

lgb_top = top_idx(lgb_sv)
xgb_top = top_idx(xgb_sv)

# ─────────────────────────────────────────────────────────────────────────────
# Figure 1 — Global importance: mean |SHAP| side by side
# ─────────────────────────────────────────────────────────────────────────────
fig_imp = make_subplots(
    rows=1, cols=2,
    subplot_titles=('LightGBM — mean |SHAP|', 'XGBoost — mean |SHAP|'),
    horizontal_spacing=0.15,
)
for col_i, (sv, top, color) in enumerate([
    (lgb_sv, lgb_top, '#4C78A8'),
    (xgb_sv, xgb_top, '#E45756'),
], start=1):
    mean_abs = np.abs(sv).mean(axis=0)[top][::-1]
    feats    = [feature_names[i] for i in top][::-1]
    fig_imp.add_trace(go.Bar(
        x=mean_abs, y=feats,
        orientation='h',
        marker_color=color,
        showlegend=False,
        hovertemplate='%{y}<br>mean |SHAP|: %{x:.4f}<extra></extra>',
    ), row=1, col=col_i)

fig_imp.update_layout(
    title='<b>SHAP Global Importance — LightGBM vs XGBoost</b><br>'
          '<sup>Mean absolute SHAP value across 2,000 test samples</sup>',
    height=520,
    margin=dict(t=90, b=40, l=180, r=40),
    plot_bgcolor='white', paper_bgcolor='white',
)
fig_imp.update_xaxes(title_text='mean |SHAP|', showgrid=True, gridcolor='#eee')
fig_imp.show()

# ─────────────────────────────────────────────────────────────────────────────
# Figure 2 & 3 — Beeswarm (LGB + XGB)
# ─────────────────────────────────────────────────────────────────────────────
def plotly_beeswarm(sv, X_data, top_indices, title):
    fig   = go.Figure()
    rng_j = np.random.default_rng(0)

    for rank, feat_i in enumerate(top_indices[::-1]):   # low rank = bottom
        vals    = sv[:, feat_i]
        fv      = X_data[:, feat_i]
        fv_norm = (fv - fv.min()) / (fv.max() - fv.min() + 1e-8)
        jitter  = rng_j.uniform(-0.35, 0.35, len(vals))

        fig.add_trace(go.Scatter(
            x=vals,
            y=rank + jitter,
            mode='markers',
            marker=dict(
                size=4,
                color=fv_norm,
                colorscale='RdBu_r',
                cmin=0, cmax=1,
                opacity=0.55,
                showscale=(rank == len(top_indices) - 1),
                colorbar=dict(
                    title='Feature<br>value',
                    tickvals=[0, 1],
                    ticktext=['Low', 'High'],
                    len=0.45, y=0.5,
                ) if rank == len(top_indices) - 1 else None,
            ),
            showlegend=False,
            hovertemplate=(
                f'<b>{feature_names[feat_i]}</b><br>'
                'SHAP: %{x:.3f}<br>'
                'Feature val: %{text}<extra></extra>'
            ),
            text=[f'{v:.3f}' for v in fv],
        ))

    fig.add_vline(x=0, line_width=1.2, line_color='grey', line_dash='dash')
    fig.update_layout(
        title=title,
        xaxis_title='SHAP value  (← lowers price  |  raises price →)',
        yaxis=dict(
            tickvals=list(range(len(top_indices))),
            ticktext=[feature_names[i] for i in top_indices[::-1]],
            showgrid=False,
        ),
        height=580,
        margin=dict(t=90, b=60, l=180, r=110),
        plot_bgcolor='white', paper_bgcolor='white',
    )
    fig.update_xaxes(showgrid=True, gridcolor='#eee', zeroline=False)
    return fig

plotly_beeswarm(
    lgb_sv, X_shap, lgb_top,
    '<b>SHAP Beeswarm — LightGBM</b><br>'
    '<sup>Each dot = one prediction  |  Color: red = high feature value, blue = low</sup>',
).show()

plotly_beeswarm(
    xgb_sv, X_shap, xgb_top,
    '<b>SHAP Beeswarm — XGBoost</b><br>'
    '<sup>Each dot = one prediction  |  Color: red = high feature value, blue = low</sup>',
).show()

# ─────────────────────────────────────────────────────────────────────────────
# Figure 4 — Dependence plots: distance & surge for both models
# ─────────────────────────────────────────────────────────────────────────────
CONT_FEATURES = ['distance', 'surge_multiplier']

fig_dep = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        'LGB — distance', 'LGB — surge_multiplier',
        'XGB — distance', 'XGB — surge_multiplier',
    ],
    vertical_spacing=0.15,
    horizontal_spacing=0.12,
)

for row_i, (sv, label) in enumerate([(lgb_sv, 'LGB'), (xgb_sv, 'XGB')], start=1):
    for col_i, feat_name in enumerate(CONT_FEATURES, start=1):
        feat_i      = feature_names.index(feat_name)
        interact    = CONT_FEATURES[1 - (col_i - 1)]          # cross-color feature
        interact_i  = feature_names.index(interact)

        fv   = X_shap[:, feat_i]
        sv_f = sv[:, feat_i]
        cv   = X_shap[:, interact_i]

        fig_dep.add_trace(go.Scatter(
            x=fv, y=sv_f,
            mode='markers',
            marker=dict(
                size=4, color=cv,
                colorscale='Viridis', opacity=0.45,
                showscale=(row_i == 1 and col_i == 2),
                colorbar=dict(title=interact, len=0.4, y=0.82),
            ),
            showlegend=False,
            hovertemplate=f'{feat_name}: %{{x:.2f}}<br>SHAP: %{{y:.3f}}<extra>{label}</extra>',
        ), row=row_i, col=col_i)

        # Binned trend line
        bin_df = (pd.DataFrame({'fv': fv, 'sv': sv_f})
                    .assign(bin=lambda d: pd.cut(d['fv'], bins=30))
                    .groupby('bin', observed=True)[['fv', 'sv']].mean()
                    .dropna())
        fig_dep.add_trace(go.Scatter(
            x=bin_df['fv'], y=bin_df['sv'],
            mode='lines',
            line=dict(color='black', width=2),
            showlegend=False,
        ), row=row_i, col=col_i)

fig_dep.add_hline(y=0, line_dash='dash', line_color='grey', line_width=1)
fig_dep.update_layout(
    title='<b>SHAP Dependence Plots</b><br>'
          '<sup>Color = interacting feature  |  Black line = binned trend  |  y=0 dashed</sup>',
    height=700,
    margin=dict(t=90, b=60, l=60, r=80),
    plot_bgcolor='white', paper_bgcolor='white',
)
fig_dep.update_xaxes(showgrid=True, gridcolor='#eee')
fig_dep.update_yaxes(title_text='SHAP value ($)', showgrid=True, gridcolor='#eee')
fig_dep.show()

# ─────────────────────────────────────────────────────────────────────────────
# Figure 5 — Waterfall: individual prediction breakdown
# ─────────────────────────────────────────────────────────────────────────────
def plotly_waterfall(sv, base_val, feat_names, sample_i, y_true, y_pred, title, total_color):
    sv_i  = sv[sample_i]

    # Top 12 features by |SHAP|, rest lumped into "Other"
    top12     = np.argsort(np.abs(sv_i))[::-1][:12]
    other_val = sv_i.sum() - sv_i[top12].sum()

    contributions = list(sv_i[top12]) + [other_val]
    labels        = [feat_names[i] for i in top12] + ['Other features']

    # Sort ascending so largest positive is at top
    order         = np.argsort(contributions)
    contributions = [contributions[i] for i in order]
    labels        = [labels[i]        for i in order]

    measures = ['absolute'] + ['relative'] * len(contributions) + ['total']
    x_vals   = [base_val]   + contributions + [base_val + sv_i.sum()]
    y_labels = [f'Base value (${base_val:.2f})'] + labels + [f'Prediction (${y_pred:.2f})']

    fig = go.Figure(go.Waterfall(
        orientation  = 'h',
        measure      = measures,
        x            = x_vals,
        y            = y_labels,
        decreasing   = dict(marker=dict(color='#4C78A8')),
        increasing   = dict(marker=dict(color='#E45756')),
        totals       = dict(marker=dict(color=total_color)),
        text         = [f'${base_val:.2f}'] +
                       [f'{v:+.2f}' for v in contributions] +
                       [f'${base_val + sv_i.sum():.2f}'],
        textposition = 'outside',
        connector    = dict(line=dict(color='#ddd', dash='dot', width=1)),
    ))
    fig.add_vline(x=base_val, line_dash='dash', line_color='grey', line_width=1)
    fig.update_layout(
        title=f'<b>{title}</b><br>'
              f'<sup>Actual: ${y_true:.2f} &nbsp;|&nbsp;'
              f' Predicted: ${y_pred:.2f} &nbsp;|&nbsp;'
              f' Error: ${abs(y_true - y_pred):.2f} &nbsp;|&nbsp;'
              f' Base (expected): ${base_val:.2f}</sup>',
        height=560,
        margin=dict(t=90, b=40, l=220, r=90),
        plot_bgcolor='white', paper_bgcolor='white',
    )
    fig.update_xaxes(title_text='Price ($)', showgrid=True, gridcolor='#eee')
    fig.update_yaxes(showgrid=False)
    return fig

SAMPLE_I = 0   # change this to inspect any prediction in the 2k sample

plotly_waterfall(
    lgb_sv, lgb_base, feature_names,
    SAMPLE_I, y_shap[SAMPLE_I], lgb_pred_shap[SAMPLE_I],
    'SHAP Waterfall — LightGBM', '#4C78A8',
).show()

plotly_waterfall(
    xgb_sv, xgb_base, feature_names,
    SAMPLE_I, y_shap[SAMPLE_I], xgb_pred_shap[SAMPLE_I],
    'SHAP Waterfall — XGBoost', '#E45756',
).show()

print(f"\nSample {SAMPLE_I} breakdown:")
print(f"  Actual:        ${y_shap[SAMPLE_I]:.2f}")
print(f"  LGB predicted: ${lgb_pred_shap[SAMPLE_I]:.2f}  (base ${lgb_base:.2f})")
print(f"  XGB predicted: ${xgb_pred_shap[SAMPLE_I]:.2f}  (base ${xgb_base:.2f})")
print(f"\nChange SAMPLE_I (0–{SHAP_N-1}) to inspect other predictions.")